### cellpose 4.0.9 · 双分支共享 ViT-L Backbone · cellprob 输出（无流场）
### v9 变更：删除阈值过滤（面积/边缘），pred_to_mask 仅保留 sigmoid + 形态学 + 连通域标记

---

## 架构

```
Phase   [B,3,H,W] → encode() → [B,256,H/8,W/8] ─┐
                    共享权重(cpsam)                 ├→ FusionBlock → decode() → [B,1,H,W]
Intensity[B,3,H,W] → encode() → [B,256,H/8,W/8] ─┘
                                  [cellprob_logit]
```

- `encode()` = patch_embed + pos_embed + ViT blocks + neck
- `FusionBlock` = concat[B,512,H/8,W/8] → Conv1×1 + GN(32) + GELU → [B,256,H/8,W/8]
- `decode()` = out + conv_transpose2d(W2) → [B,1,H,W]

## 数据目录结构

```
seg_train/                           seg_test/（独立测试集，同结构）
└── 药物条件/时间点/images_folder/
    ├── 01_phase27.tiff              ← 相位图（RGB 彩色）
    ├── 01_intensity27.tiff          ← 明场图（灰度存为 RGB）
    └── 01_mask27.png                ← 语义标签：0=背景 1=单体 2000=聚集体
```

## 完整输出目录结构（v6）

```
OUTPUT_DIR/<exp_id>/
├── eval/                               ← ★ Step 7 测试集评估结果（有 GT 对比）
│   ├── test_results.csv                  AP@0.5/0.75/0.9 + count_bias/mae/mape
│   ├── test_results_stratified.csv       按药物/时间点分层 + 高风险标记
│   ├── failure_analysis.csv              AP@0.5 < 0.3 失败案例
│   └── <subdir>/
│       ├── <stem>_compare.png            6列对比图（含 GT）
│       └── <stem>_pred_mask.tiff         实例 mask uint16
│
└── inference/                          ← ★ Step 9 批量推理（无需 GT）
    ├── batch_summary.csv                 细胞计数汇总
    └── <subdir>/
        ├── <stem>_compare.png            6列对比图（无 GT 时为 5列）
        └── <stem>_pred_mask.tiff         实例 mask uint16

CKPT_DIR/<exp_id>/
├── best.pth                            ← 验证集最优 checkpoint
├── latest.pth                          ← 每 epoch 覆盖，断点续训
├── interrupt.pth                       ← Ctrl+C 中断保存（含 exp_id）
└── epoch_XXX_warmup_end.pth            ← warmup 结束关键节点

LOG_DIR/
├── experiments_registry.csv            ← 全局实验对比总表
└── <exp_id>/
    ├── config.json                      ← 超参数快照（续训追加）
    ├── train_log.csv                    ← epoch/tr_prob/tr_dice/val_prob/val_dice/lr
    ├── training_curves.png              ← BCE + Dice 双曲线图
    ├── training_summary.json            ← 训练结束汇总
    └── dataset_stats.json               ← 数据集统计快照
```

## v8 对比图（5/4列说明）

```
列1  Phase (RGB)      : 相位图原图
列2  Intensity (Gray) : 明场图原图
列3  Cell Probability : cellprob 热力图（越亮=越可能是细胞）
列4  Predicted Masks  : 阈值+形态学过滤后的最终实例掩膜（随机彩色）
列5  GT Masks         : Ground Truth（有 GT 时显示，否则省略 → 共 4 列）
```

## train_log.csv 列说明（v6 新增 Dice 列）

```
epoch     训练轮次
tr_prob   训练集 BCE loss
tr_dice   训练集 Dice loss
val_prob  验证集 BCE loss
val_dice  验证集 Dice loss
lr        当前学习率
```

from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ── 安装依赖 ────────────────────────────────────────────────────────
# cellpose 4.x 需要从 GitHub 最新源安装，pip 版本可能过旧
import subprocess
subprocess.run(["pip","install","-q","git+https://github.com/mouseland/cellpose.git"], check=True)
subprocess.run(["pip","install","-q","segmentation_models_pytorch","six","pandas",
                "tifffile","scikit-image","scikit-learn","tqdm"], check=True)

import importlib.metadata, torch
print(f"cellpose : {importlib.metadata.version('cellpose')}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Step 0.5 — 下载 cpsam 预训练权重

cellpose 官方 `cpsam`（SAM/ViT-L），路径固定为 `/root/.cellpose/models/cpsam`。
首次运行约需下载 1.2 GB，已下载则跳过。


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ── 安装依赖 ────────────────────────────────────────────────────────
# cellpose 4.x 需要从 GitHub 最新源安装，pip 版本可能过旧
import subprocess
subprocess.run(["pip","install","-q","git+https://github.com/mouseland/cellpose.git"], check=True)
subprocess.run(["pip","install","-q","segmentation_models_pytorch","six","pandas",
                "tifffile","scikit-image","scikit-learn","tqdm"], check=True)

import importlib.metadata, torch
print(f"cellpose : {importlib.metadata.version('cellpose')}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import os
from cellpose import models as cp_models

CPSAM_PATH = os.path.expanduser('~/.cellpose/models/cpsam')

if os.path.exists(CPSAM_PATH):
    print(f'cpsam 已存在: {CPSAM_PATH}')
    print(f'文件大小: {os.path.getsize(CPSAM_PATH)/1e6:.1f} MB')
else:
    print('正在下载 cpsam 权重（约 1.2 GB，请耐心等待）...')
    _tmp = cp_models.CellposeModel(model_type='cpsam', gpu=False)
    del _tmp
    if os.path.exists(CPSAM_PATH):
        print(f'✅ 下载完成: {CPSAM_PATH}')
    else:
        print('[ERROR] 下载失败，请检查网络连接')

import torch
raw = torch.load(CPSAM_PATH, map_location='cpu')
keys = list(raw.keys())[:5] if isinstance(raw, dict) else '(非 dict 格式)'
print(f'权重格式验证: top-level keys = {keys}')
print('cpsam 权重就绪 ✓')


Mounted at /content/drive
cellpose : 4.1.1
PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


正在下载 cpsam 权重（约 1.2 GB，请耐心等待）...


100%|██████████| 1.15G/1.15G [00:06<00:00, 191MB/s]


✅ 下载完成: /root/.cellpose/models/cpsam
权重格式验证: top-level keys = ['W2', 'encoder.pos_embed', 'encoder.patch_embed.proj.weight', 'encoder.patch_embed.proj.bias', 'encoder.blocks.0.norm1.weight']
cpsam 权重就绪 ✓


## Step 1 — 全局配置

★ 修改区域：所有路径和超参数在此处集中配置。

**v6 新增参数：**
- `LAMBDA_BCE` / `LAMBDA_DICE`：联合损失权重
- 输出目录自动按 eval/ 和 inference/ 区分


In [ ]:
import os, logging
logging.getLogger('cellpose').setLevel(logging.ERROR)

# ══════════════════════════════════════════════════════════════
# ★ 修改区域 ★
# ══════════════════════════════════════════════════════════════
MODEL_PATH = os.path.expanduser('~/.cellpose/models/cpsam')

TRAIN_ROOT = '/content/drive/MyDrive/drug_effect/seg_train'
TEST_ROOT  = '/content/drive/MyDrive/drug_effect/seg_test'
OUTPUT_DIR = '/content/drive/MyDrive/drug_effect/seg/outputs'
CKPT_DIR   = '/content/drive/MyDrive/drug_effect/seg/ckpts'
LOG_DIR    = '/content/drive/MyDrive/drug_effect/seg/logs'

# 文件命名关键字
PHASE_KEYWORD     = '_phase'
INTENSITY_KEYWORD = '_intensity'
MASK_KEYWORD      = '_mask'
PHASE_EXT         = '.tiff'
INTENSITY_EXT     = '.tiff'
MASK_EXT          = '.png'

# 实验管理
EXP_NAME      = ''  # 可选备注，留空则纯时间戳
RESUME_EXP_ID = ''             # 同一实验断点续训时填写；新实验请留空

FINETUNE_FROM_EXP_ID = ''  # ★ 迁移权重（新实验 + 旧权重）
EVAL_EXP_ID  = '20260326_103536'   # 指定评估测试集所用best.pth的实验
INFER_EXP_ID = '20260326_103536'   # ★ v7 新增：指定批量推理所用best.pth的实验；留空→沿用Step7/5的exp_id

# 变更说明（追加写入 config.json，用于多次续训后的参数溯源）
CHANGE_NOTE = ''

# ── 训练参数 ──────────────────────────────────────────────────
BSIZE           = 256      # ViT-L tile 尺寸（训练裁剪 & 滑窗 tile 尺寸）
SW_STRIDE_RATIO = 0.25     # 滑动窗口步长比例：0.25 → stride=192px，629×630 只需 2×2=4 tiles
BATCH_SIZE      = 160
EPOCHS          = 50      # ★ 100 → 120
LR_BACKBONE     = 1e-5     # ★ 1e-5 → 5e-6：backbone 已充分训练，防止 ep57 spike
LR_FUSION       = 1e-3     # ★ 1e-3 → 5e-4：FusionBlock 60轮后无需大 lr
WEIGHT_DECAY    = 1e-4
VAL_RATIO       = 0.1
SEED            = 42
NUM_WORKERS     = 8
PATIENCE        = 15       # ★ 10 → 15：给 Dice 更多收敛时间

# ── ★ 启动训练 ─────────────────────────────────────────────────────
WARMUP_FREEZE = False      # ★ 迁移权重时无需 warmup
WARMUP_EPOCHS = 5
RESUME = False             # ★ 新实验不续训旧 epoch 状态

# ── v6 新增：联合损失权重 ──────────────────────────────────────
# total_loss = LAMBDA_BCE × BCE + LAMBDA_DICE × Dice
# · BCE  擅长逐像素二分类，收敛快，对类别不平衡敏感
# · Dice 直接优化 F1 分数，对前景/背景比例不敏感（血小板占比小时很有用）
# · 两者互补：BCE 稳定训练，Dice 强化分割边界质量
LAMBDA_BCE  = 1.0          # ★ 1.0 → 0.5：BCE 已收敛，减少优化压力
LAMBDA_DICE = 1.0          # ★ 1.0 → 2.0：Dice 仍有改善空间，重点优化边界

# ── 推理参数 ──────────────────────────────────────────────────
PROB_THRESHOLD = 0.5    # cellprob sigmoid 后的阈值（对应 logit=0.0）
DIAMETER       = 90.0   # 细胞直径（px），用于图算法距离阈值
# ── 后处理参数 ─────────────────────────────────────────────────
# v9：删除面积过滤（MIN_CELL_AREA_PX）和边缘排除（MARGIN），
#      后处理仅保留 PROB_THRESHOLD + 形态学操作。
# ══════════════════════════════════════════════════════════════

for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("=" * 65)
print("  Dual-Branch Cellpose v6 (SAM/ViT-L) — BCE+Dice 联合损失")
print("=" * 65)
print(f"  MODEL_PATH   : {MODEL_PATH}")
print(f"  exists       : {os.path.exists(MODEL_PATH)}")
print(f"  TRAIN_ROOT   : {TRAIN_ROOT}")
print(f"  BSIZE        : {BSIZE}  BATCH: {BATCH_SIZE}")
print(f"  LR backbone  : {LR_BACKBONE}  LR fusion: {LR_FUSION}")
print(f"  LAMBDA_BCE   : {LAMBDA_BCE}   LAMBDA_DICE: {LAMBDA_DICE}")
print(f"  Epochs       : {EPOCHS}  Patience: {PATIENCE}")
print(f"  PROB_THRESH  : {PROB_THRESHOLD}")
print("=" * 65)


  Dual-Branch Cellpose v6 (SAM/ViT-L) — BCE+Dice 联合损失
  MODEL_PATH   : /root/.cellpose/models/cpsam
  exists       : True
  TRAIN_ROOT   : /content/drive/MyDrive/drug_effect/seg_train
  BSIZE        : 256  BATCH: 160
  LR backbone  : 1e-05  LR fusion: 0.001
  LAMBDA_BCE   : 1.0   LAMBDA_DICE: 1.0
  Epochs       : 50  Patience: 15
  PROB_THRESH  : 0.5


## Step 2 — Dataset

- 递归扫描三层嵌套目录，自动配对 phase / intensity / mask
- GT mask 值：`0`=背景，`1`=单体，`2000`=聚集体（语义标签，训练时转二值概率图）
- 数据增强：随机裁剪 + 水平/垂直翻转 + 90°旋转 + 亮度/对比度抖动


In [ ]:
import random, csv, time
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import cv2
import numpy as np
import tifffile
import torch
from torch.utils.data import Dataset, DataLoader
from cellpose import transforms as cp_tf
import re


def make_stem(filename: str) -> str:
    """01_phase45.tiff → '01045'（前缀2位 + 后缀3位）"""
    m = re.match(r'^(\d+)_phase(\d+)', filename)
    if m:
        return m.group(1).zfill(2) + m.group(2).zfill(3)
    return filename.replace(PHASE_KEYWORD, '').replace(PHASE_EXT, '')


def scan_pairs(root_dir: str, require_mask: bool = True) -> List[Dict]:
    """递归扫描目录，返回 phase/intensity/mask 三元组列表"""
    root, samples = Path(root_dir), []
    for phase_path in sorted(root.rglob(f'*{PHASE_KEYWORD}*{PHASE_EXT}')):
        name     = phase_path.name
        int_path = phase_path.parent / name.replace(PHASE_KEYWORD, INTENSITY_KEYWORD)
        if not int_path.exists(): continue
        mask_name = name.replace(PHASE_KEYWORD, MASK_KEYWORD).replace(PHASE_EXT, MASK_EXT)
        mask_path = phase_path.parent / mask_name
        has_mask  = mask_path.exists()
        if require_mask and not has_mask: continue
        rel = str(phase_path.relative_to(root).parent)
        samples.append({
            'phase'    : str(phase_path),
            'intensity': str(int_path),
            'mask'     : str(mask_path) if has_mask else None,
            'stem'     : make_stem(name),
            'subdir'   : rel,
        })
    n_mask = sum(1 for s in samples if s['mask'])
    print(f'  [{Path(root_dir).name}] Found {len(samples)} pairs ({n_mask} with masks)')
    return samples


def split_train_val(samples, val_ratio=VAL_RATIO, seed=SEED):
    rng = random.Random(seed)
    s = samples.copy(); rng.shuffle(s)
    n_v = int(len(s) * val_ratio)
    val, train = s[:n_v], s[n_v:]
    print(f'  Split → train={len(train)}  val={len(val)}')
    return train, val


def read_rgb(path: str) -> np.ndarray:
    img = cv2.imread(path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = tifffile.imread(path)
        if img.ndim == 2: img = np.stack([img]*3, axis=-1)
        elif img.ndim == 3 and img.shape[0] in (1,3,4): img = img.transpose(1,2,0)
        if img.shape[-1] > 3: img = img[..., :3]
    return img.astype(np.uint8)


def read_mask(path: Optional[str]) -> Optional[np.ndarray]:
    if path is None or not Path(path).exists(): return None
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: img = tifffile.imread(path)
    return img.astype(np.uint32)


def normalize_cp(img_hwc: np.ndarray) -> np.ndarray:
    return cp_tf.normalize_img(img_hwc.astype(np.float32))


def _elastic_transform(image: np.ndarray, alpha: float, sigma: float,
                        rng: np.random.RandomState) -> np.ndarray:
    """
    弹性形变核心实现（内部函数，由 sync_augment 调用）。

    原理：用高斯平滑后的随机位移场对图像像素做双线性重采样，
    模拟细胞受压变形、聚集体形状不规则等真实形态变化。

    Args:
        image : [H, W] 或 [H, W, C]  uint8
        alpha : 位移幅度（px）。推荐 15-25；过大会扭曲到无法识别
        sigma : 高斯平滑 σ（px）。推荐 4-6；越大变形越平滑
        rng   : 已初始化的 RandomState，确保 phase/intens/mask 三者共用
                同一位移场（空间一致性）
    """
    from scipy.ndimage import gaussian_filter, map_coordinates
    H, W = image.shape[:2]
    # 生成随机位移场并高斯平滑（平滑 → 局部连续变形，不产生折叠）
    dx = gaussian_filter(rng.randn(H, W), sigma) * alpha
    dy = gaussian_filter(rng.randn(H, W), sigma) * alpha
    x, y = np.meshgrid(np.arange(W), np.arange(H))
    coords = [np.clip(y + dy, 0, H - 1).ravel(),
              np.clip(x + dx, 0, W - 1).ravel()]
    if image.ndim == 3:
        return np.stack(
            [map_coordinates(image[..., c], coords, order=1).reshape(H, W)
             for c in range(image.shape[2])], axis=-1
        ).astype(image.dtype)
    return map_coordinates(image, coords, order=1).reshape(H, W).astype(image.dtype)


def sync_augment(phase, intens, mask, crop=BSIZE):
    """
    训练时同步数据增强 — 精简为 10× 扩增的最优三步组合。

    ┌─────────────────────────────────────────────────────────────────┐
    │  设计原则：只保留对血小板分割性价比最高的三类增强，              │
    │  去除 Zoom / Cutout / 噪声等对 10× 目标贡献边际较低的方法。    │
    └─────────────────────────────────────────────────────────────────┘

    流水线（按顺序执行）：

    Step 1 ── 空间翻转 + 旋转（必定执行，贡献 8 个确定性变体）
              H-flip × V-flip × 4旋转 = 8 种无损几何变换
              零信息损失，直接覆盖所有朝向，是 10× 的主要来源。

    Step 2 ── 随机裁剪（必定执行）
              从 reflect-pad 后的原图随机裁 BSIZE×BSIZE 区域。
              全图 630×629 ≈ 2.5× tile，每次取不同位置增加位置多样性。

    Step 3 ── 弹性形变（概率 p=0.4）
              模拟血小板受压变形、聚集体形状不规则。
              参数：α∈[15,25]px  σ∈[4,6]px（局部平滑，不产生折叠）
              phase / intens / mask 共用同一位移场，保证标注对齐。
              选择依据：直接增加形态多样性，对聚集体/单体分类帮助最大。

    Step 4 ── 逐通道独立亮度/对比度（概率 p=0.3）
              模拟不同批次、不同成像参数下的照明差异。
              phase 和 intens 独立采样（不相关，更真实）。

              相位图（三通道独立）：
                R/G 通道：α∈[0.85,1.15]  β∈[-8,+8]
                B 通道  ：α∈[0.88,1.10]  β∈[-6,+6]
                ← B 通道含环形晕圈物理结构，范围收窄防止梯度被压缩

              明场图（三通道同步，因为三通道完全相同）：
                α∈[0.85,1.20]  β∈[-12,+12]
                ← 明场对比度变化空间更大，范围适当放宽

    排除方法说明：
      · Zoom 缩放    — 10× 目标下翻转/旋转已足够，额外缩放收益小
      · Cutout       — 血小板信号区域小（约 5%），遮挡极易破坏关键边界
      · 高斯噪声     — 相位背景已有自然颗粒感，人工噪声引入模式混淆
      · CLAHE        — 改变局部对比度会破坏相位 B 通道环形晕圈物理特征
    """
    H, W = phase.shape[:2]

    # ── Step 1. 空间翻转 + 旋转 ─────────────────────────────────────
    if random.random() > 0.5:                          # 垂直翻转
        phase = phase[::-1].copy()
        intens = intens[::-1].copy()
        if mask is not None: mask = mask[::-1].copy()
    if random.random() > 0.5:                          # 水平翻转
        phase = phase[:, ::-1].copy()
        intens = intens[:, ::-1].copy()
        if mask is not None: mask = mask[:, ::-1].copy()
    k = random.randint(0, 3)                           # 90° × k 旋转
    if k:
        phase = np.rot90(phase, k).copy()
        intens = np.rot90(intens, k).copy()
        if mask is not None: mask = np.rot90(mask, k).copy()

    # ── Step 2. 随机裁剪 crop×crop ──────────────────────────────────
    H, W = phase.shape[:2]
    if H < crop or W < crop:
        ph_pad = max(0, crop - H); pw_pad = max(0, crop - W)
        phase  = np.pad(phase,  ((0, ph_pad), (0, pw_pad), (0, 0)), mode='reflect')
        intens = np.pad(intens, ((0, ph_pad), (0, pw_pad), (0, 0)), mode='reflect')
        if mask is not None:
            mask = np.pad(mask, ((0, ph_pad), (0, pw_pad)), mode='constant')
        H, W = phase.shape[:2]
    y0 = random.randint(0, H - crop)
    x0 = random.randint(0, W - crop)
    phase  = phase [y0:y0+crop, x0:x0+crop]
    intens = intens[y0:y0+crop, x0:x0+crop]
    if mask is not None: mask = mask[y0:y0+crop, x0:x0+crop]

    # ── Step 3. 弹性形变（p=0.4）────────────────────────────────────
    # phase / intens / mask 共用同一位移场（保证标注严格对齐）
    if random.random() < 0.4:
        alpha = random.uniform(15.0, 25.0)   # 位移幅度（px）
        sigma = random.uniform(4.0,  6.0)    # 高斯平滑 σ（px）
        # 用固定 seed 生成位移场，三者复用
        _seed = random.randint(0, 2**31)
        rng   = np.random.RandomState(_seed)
        phase  = _elastic_transform(phase,  alpha, sigma, rng)
        rng    = np.random.RandomState(_seed)   # 重置 → 相同位移场
        intens = _elastic_transform(intens, alpha, sigma, rng)
        if mask is not None:
            rng  = np.random.RandomState(_seed)
            mask = _elastic_transform(mask, alpha, sigma, rng)

    # ── Step 4. 逐通道独立亮度/对比度（p=0.5）──────────────────────
    if random.random() < 0.5:
        # 相位图：R/G 宽范围 + B 通道收窄（保护环形晕圈梯度结构）
        ph_f = phase.astype(np.float32)
        for c, (a_lo, a_hi, b_rng) in enumerate([
            (0.85, 1.15, 8.0),   # R：信号弱，范围稍宽
            (0.85, 1.15, 8.0),   # G：中等信号，同 R
            (0.88, 1.10, 6.0),   # B：环形晕圈，α 上限压至 1.10，β 收窄
        ]):
            a = random.uniform(a_lo, a_hi)
            b = random.uniform(-b_rng, b_rng)
            ph_f[..., c] = ph_f[..., c] * a + b
        phase = np.clip(ph_f, 0, 255).astype(np.uint8)

        # 明场图：三通道完全相同（灰度图），统一采样即可
        it_alpha = random.uniform(0.85, 1.20)
        it_beta  = random.uniform(-12.0, 12.0)
        intens   = np.clip(intens.astype(np.float32) * it_alpha + it_beta,
                           0, 255).astype(np.uint8)

    # 2. 微小伽玛校正 (p=0.3) - 模拟微弱的光照非线性响应
    if random.random() < 0.3:
        # γ 严格控制在 [0.9, 1.1] 之间，防止破坏 B 通道结构
        gamma = random.uniform(0.9, 1.1)
        inv_gamma = 1.0 / gamma

        # 预计算查找表 (LUT) 以提高速度
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        phase = cv2.LUT(phase, table)
        intens = cv2.LUT(intens, table)

    return phase, intens, mask


def center_crop(arr, crop):
    H, W = arr.shape[:2]
    if H < crop or W < crop:
        ph, pw = max(0,crop-H), max(0,crop-W)
        arr = np.pad(arr,
                     ((0,ph),(0,pw),(0,0)) if arr.ndim==3 else ((0,ph),(0,pw)),
                     mode='reflect' if arr.ndim==3 else 'constant')
        H, W = arr.shape[:2]
    y0, x0 = (H-crop)//2, (W-crop)//2
    return arr[y0:y0+crop, x0:x0+crop]


def sliding_window_inference(eval_model, ph_arr, it_arr,
                              tile_size=None, stride_ratio=None, device=None):
    """
    滑动窗口推理：在完整原图上逐块推理并平均，保留所有血小板完整边界。

    Args:
        eval_model  : 推理模型（eval 状态）
        ph_arr      : [H, W, 3] uint8  相位图（完整原图，无需裁剪）
        it_arr      : [H, W, 3] uint8  明场图
        tile_size   : 单块尺寸，默认 BSIZE
        stride_ratio: 重叠比例，0.25 → 75% 覆盖，629×630 只需 2×2=4 tiles
        device      : torch device

    Returns:
        logit_full  : [1, H, W] float32  cellprob logit
        ph_arr      : 原始相位图（直接透传）
        it_arr      : 原始明场图（直接透传）
    """
    if tile_size    is None: tile_size    = BSIZE
    if stride_ratio is None: stride_ratio = SW_STRIDE_RATIO
    if device       is None: device       = next(eval_model.parameters()).device

    H_orig, W_orig = ph_arr.shape[:2]
    stride = max(1, int(tile_size * (1 - stride_ratio)))

    pad_h = max(tile_size - H_orig, 0)
    pad_w = max(tile_size - W_orig, 0)
    if H_orig > tile_size:
        extra_h = (stride - (H_orig - tile_size) % stride) % stride
        pad_h += extra_h
    if W_orig > tile_size:
        extra_w = (stride - (W_orig - tile_size) % stride) % stride
        pad_w += extra_w

    ph_pad = np.pad(ph_arr, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    it_pad = np.pad(it_arr, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    H_pad, W_pad = ph_pad.shape[:2]

    logit_acc  = np.zeros((H_pad, W_pad), dtype=np.float64)
    weight_acc = np.zeros((H_pad, W_pad), dtype=np.float64)

    # 高斯权重窗：边缘权重低→中心权重高，抑制 tile 拼接伪影
    sigma = tile_size / 8.0
    cy, cx = np.mgrid[0:tile_size, 0:tile_size]
    gauss_w = np.exp(-((cy-tile_size/2)**2 + (cx-tile_size/2)**2) / (2*sigma**2)).astype(np.float64)

    ys = list(range(0, H_pad - tile_size + 1, stride))
    xs = list(range(0, W_pad - tile_size + 1, stride))

    eval_model.eval()
    with torch.no_grad():
        for y0 in ys:
            for x0 in xs:
                ph_tile = ph_pad[y0:y0+tile_size, x0:x0+tile_size]
                it_tile = it_pad[y0:y0+tile_size, x0:x0+tile_size]
                p_t = (torch.from_numpy(normalize_cp(ph_tile).transpose(2,0,1))
                       .unsqueeze(0).to(device))
                i_t = (torch.from_numpy(normalize_cp(it_tile).transpose(2,0,1))
                       .unsqueeze(0).to(device))
                out, _ = eval_model(p_t, i_t)
                tile_logit = out.float().cpu().numpy()[0, 0]
                logit_acc [y0:y0+tile_size, x0:x0+tile_size] += tile_logit * gauss_w
                weight_acc[y0:y0+tile_size, x0:x0+tile_size] += gauss_w

    logit_full = (logit_acc / np.maximum(weight_acc, 1e-8))[:H_orig, :W_orig]
    logit_full = logit_full[np.newaxis].astype(np.float32)
    return logit_full, ph_arr, it_arr


class PlateletDataset(Dataset):
    """
    训练/验证用 Dataset。
    - train 模式：随机裁剪 + 数据增强
    - val   模式：中心裁剪（固定 BSIZE，用于批量验证）
    """
    def __init__(self, samples: List[Dict], mode: str = 'train'):
        self.samples = samples
        self.mode    = mode

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        ph  = read_rgb(s['phase'])
        it  = read_rgb(s['intensity'])
        msk = read_mask(s['mask'])

        if self.mode == 'train':
            ph, it, msk = sync_augment(ph, it, msk)
        else:
            ph  = center_crop(ph, BSIZE)
            it  = center_crop(it, BSIZE)
            if msk is not None: msk = center_crop(msk, BSIZE)

        ph_t  = torch.from_numpy(normalize_cp(ph).transpose(2,0,1))
        it_t  = torch.from_numpy(normalize_cp(it).transpose(2,0,1))
        msk_t = torch.from_numpy(msk.astype(np.int32)) if msk is not None \
                else torch.zeros(BSIZE, BSIZE, dtype=torch.int32)
        return {'phase': ph_t, 'intensity': it_t, 'mask': msk_t}


# 扫描数据集
print('Scanning datasets...')
train_all    = scan_pairs(TRAIN_ROOT)
test_samples = scan_pairs(TEST_ROOT, require_mask=False)
train_samples, val_samples = split_train_val(train_all)

train_loader = DataLoader(PlateletDataset(train_samples, 'train'),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(PlateletDataset(val_samples, 'val'),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f'\nDataLoader ready:  train={len(train_loader)} batches | val={len(val_loader)} batches')


Scanning datasets...
  [seg_train] Found 500 pairs (500 with masks)
  [seg_test] Found 200 pairs (200 with masks)
  Split → train=450  val=50

DataLoader ready:  train=2 batches | val=1 batches


## Step 3 — 双分支模型

继承 cellpose v4 `Transformer`，覆写 `forward()` 实现双分支融合。
FusionBlock 随机初始化，其余权重从 cpsam 加载。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
from cellpose.vit_sam import Transformer
import numpy as np


class FusionBlock(nn.Module):
    """
    concat [B,512,H,W] → Conv1×1 + GN(32) + GELU → [B,256,H,W]
    随机初始化，使用大 lr 快速收敛。
    """
    def __init__(self, channels=256):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv2d(channels*2, channels, kernel_size=1, bias=False),
            nn.GroupNorm(32, channels),
            nn.GELU(),
        )
        nn.init.kaiming_normal_(self.fuse[0].weight, mode='fan_out', nonlinearity='relu')

    def forward(self, fp, fi):
        return self.fuse(torch.cat([fp, fi], dim=1))


class DualBranchTransformer(Transformer):
    """
    双分支共享 SAM/ViT-L 血小板分割模型（概率图版，无流场）。

    输入: phase / intensity  [B, 3, H, W]
    输出: (cellprob_logit [B,1,H,W], style [B,256])
    """
    def __init__(self, pretrained_path=None, backbone='vit_l',
                 ps=8, nout=1, bsize=256, rdrop=0.4, dtype=torch.float32):
        super().__init__(backbone=backbone, ps=ps, nout=nout,
                         bsize=bsize, rdrop=rdrop, checkpoint=None, dtype=dtype)
        self.fusion = FusionBlock(channels=256)
        if pretrained_path:
            if os.path.exists(pretrained_path):
                self._load_weights(pretrained_path)
            else:
                raise FileNotFoundError(
                    f'权重文件不存在: {pretrained_path}\n请先运行 Step 0.5')
        else:
            print('[INFO] 未指定权重路径，随机初始化。')

    def _load_weights(self, path: str):
        """加载 cpsam 权重，兼容 state_dict / {'model':...} / DataParallel 格式"""
        raw = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(raw, dict) and 'model' in raw:     sd = raw['model']
        elif isinstance(raw, dict) and 'state_dict' in raw: sd = raw['state_dict']
        else:                                             sd = raw
        if isinstance(sd, dict) and any(k.startswith('module.') for k in sd):
            sd = OrderedDict({k[7:]: v for k, v in sd.items()})
        own_sd = self.state_dict()
        SKIP_KEYS = {'out.weight', 'out.bias', 'W2'}
        filtered  = OrderedDict()
        for k, v in sd.items():
            if k in SKIP_KEYS and k in own_sd and own_sd[k].shape != v.shape:
                print(f'  [Weights] skip {k}: ckpt{list(v.shape)} → model{list(own_sd[k].shape)}')
                continue
            filtered[k] = v
        result   = self.load_state_dict(filtered, strict=False)
        n_total  = len(self.state_dict())
        n_fusion = len([k for k in result.missing_keys if 'fusion' in k])
        n_other  = len(result.missing_keys) - n_fusion
        n_loaded = n_total - len(result.missing_keys)
        print(f'  [Weights] {n_loaded}/{n_total} params loaded ({100.*n_loaded/max(n_total,1):.1f}%)')
        print(f'  [Weights] FusionBlock missing (expected): {n_fusion}')
        if n_other > 0:
            print(f'  [Weights] ⚠ Other missing ({n_other}): '
                  f'{[k for k in result.missing_keys if "fusion" not in k][:5]}')
        else:
            print(f'  [Weights] ✓ 预训练权重完整加载')

    def encode(self, x):
        """patch_embed + pos_embed（双线性插值支持任意分辨率）+ ViT blocks + neck"""
        x = self.encoder.patch_embed(x)
        if self.encoder.pos_embed is not None:
            pe = self.encoder.pos_embed
            if pe.shape[1:3] != x.shape[1:3]:
                pe = F.interpolate(
                    pe.permute(0,3,1,2).float(),
                    size=x.shape[1:3], mode='bilinear', align_corners=False
                ).to(x.dtype).permute(0,2,3,1)
            x = x + pe
        if self.training and self.rdrop > 0:
            nlay  = len(self.encoder.blocks)
            drops = (torch.rand((len(x), nlay), device=x.device) <
                     torch.linspace(0, self.rdrop, nlay, device=x.device)).to(x.dtype)
            for i, blk in enumerate(self.encoder.blocks):
                mask = drops[:, i].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
                if not next(blk.parameters()).requires_grad:
                    x = blk(x)
                else:
                    x = x * mask + torch.utils.checkpoint.checkpoint(
                        blk, x, use_reentrant=False) * (1 - mask)
        else:
            for blk in self.encoder.blocks:
                x = blk(x)
        x = self.encoder.neck(x.permute(0,3,1,2))
        return x

    def decode(self, feat):
        """readout head: [B,256,H/ps,W/ps] → [B,1,H,W]"""
        x = self.out(feat)
        x = F.conv_transpose2d(x, self.W2, stride=self.ps, padding=0)
        return x

    def forward(self, phase, intensity):
        feat_p = self.encode(phase)
        feat_i = self.encode(intensity)
        fused  = self.fusion(feat_p, feat_i)
        out    = self.decode(fused)
        style  = torch.zeros((phase.shape[0], 256), device=phase.device, dtype=out.dtype)
        return out, style

    def get_param_groups(self, lr_backbone=LR_BACKBONE, lr_fusion=LR_FUSION):
        pretrained = list(self.encoder.parameters()) + list(self.out.parameters())
        return [
            {'params': pretrained,                     'lr': lr_backbone, 'name': 'backbone'},
            {'params': list(self.fusion.parameters()),  'lr': lr_fusion,  'name': 'fusion'},
        ]

    def freeze_backbone(self):
        for p in self.encoder.parameters(): p.requires_grad = False
        for p in self.out.parameters():     p.requires_grad = False
        print('[Model] Backbone frozen → only FusionBlock training.')

    def unfreeze_backbone(self):
        for p in self.parameters(): p.requires_grad = True
        print('[Model] All parameters unfrozen → full fine-tuning.')

    def count_params(self):
        def _n(ps): return sum(p.numel() for p in ps)
        return {'encoder'  : _n(self.encoder.parameters()),
                'out+W2'   : _n(self.out.parameters()),
                'fusion'   : _n(self.fusion.parameters()),
                'total'    : _n(self.parameters()),
                'trainable': sum(p.numel() for p in self.parameters() if p.requires_grad)}


print('DualBranchTransformer defined ✓')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = DualBranchTransformer(
    pretrained_path=MODEL_PATH,
    dtype=torch.float32,
).to(device)

pc = model.count_params()
print(f'\n参数统计:')
print(f'  encoder (ViT-L)  : {pc["encoder"]:>14,}')
print(f'  out + W2         : {pc["out+W2"]:>14,}')
print(f'  FusionBlock(新增): {pc["fusion"]:>14,}  ← 随机初始化')
print(f'  TOTAL            : {pc["total"]:>14,}')
print(f'  trainable        : {pc["trainable"]:>14,}')

# 前向测试
model.eval()
with torch.no_grad():
    p_d = torch.randn(1, 3, BSIZE, BSIZE, device=device)
    i_d = torch.randn(1, 3, BSIZE, BSIZE, device=device)
    out, style = model(p_d, i_d)
print(f'\n前向测试: output [cellprob]: {out.shape}  ✓')


DualBranchTransformer defined ✓
Device: cuda
  [Weights] skip W2: ckpt[192, 3, 8, 8] → model[64, 1, 8, 8]
  [Weights] skip out.weight: ckpt[192, 256, 1, 1] → model[64, 256, 1, 1]
  [Weights] skip out.bias: ckpt[192] → model[64]
  [Weights] 345/353 params loaded (97.7%)
  [Weights] FusionBlock missing (expected): 3
  [Weights] ⚠ Other missing (5): ['W2', 'diam_labels', 'diam_mean', 'out.weight', 'out.bias']

参数统计:
  encoder (ViT-L)  :    304,542,720
  out + W2         :         16,448
  FusionBlock(新增):        131,584  ← 随机初始化
  TOTAL            :    304,694,850
  trainable        :    304,690,752

前向测试: output [cellprob]: torch.Size([1, 1, 256, 256])  ✓


## Step 4 — 损失函数（BCE + Dice 联合）

**v6 新增 Dice Loss：**

- `BCE`：逐像素二分类，收敛稳定，对背景主导图像敏感
- `Dice`：直接优化 F1-score，对前景/背景不平衡鲁棒（血小板占比通常 < 5%）
- 联合：`total = λ_bce × BCE + λ_dice × Dice`

损失分项均记录到 train_log.csv（`tr_prob` / `tr_dice` / `val_prob` / `val_dice`）。


In [ ]:
from scipy import ndimage


def gt_to_prob(masks_np: np.ndarray) -> np.ndarray:
    """
    GT mask → 二值概率图，用于 BCE/Dice loss。
    masks_np: [B, H, W]，值 0=背景 / 1=单体 / 2000=聚集体
    返回: float32，前景=1.0 背景=0.0
    """
    return (masks_np > 0).astype(np.float32)


class CellposeLoss(nn.Module):
    """
    BCE + Dice 联合损失。

    BCE：BCEWithLogitsLoss，对 logit 直接计算，数值稳定。
    Dice：Soft-Dice，通过 sigmoid 把 logit 转概率后计算：
          Dice = 1 - (2 * Σ(p*g) + ε) / (Σp + Σg + ε)
          smooth=1.0 防止分母为零（空图时也稳定）。

    pred     : [B, 1, H, W]  cellprob logit（未经 sigmoid）
    gt_probs : [B, H, W]     0/1 二值图，float32
    返回 dict : total / prob(BCE) / dice
    """
    def __init__(self, lambda_bce: float = LAMBDA_BCE,
                       lambda_dice: float = LAMBDA_DICE):
        super().__init__()
        self.bce        = nn.BCEWithLogitsLoss()
        self.lambda_bce  = lambda_bce
        self.lambda_dice = lambda_dice

    @staticmethod
    def dice_loss(pred_logit: torch.Tensor,
                  gt: torch.Tensor,
                  smooth: float = 1.0) -> torch.Tensor:
        """
        Soft-Dice loss（在 sigmoid 概率空间计算）。
        pred_logit : [B, H, W]  未经 sigmoid 的 logit
        gt         : [B, H, W]  0/1 float32
        """
        pred_prob    = torch.sigmoid(pred_logit)               # → 概率 (0,1)
        # 在 H×W 维度求和，保留 batch 维度
        intersection = (pred_prob * gt).sum(dim=(-2, -1))      # [B]
        union        = pred_prob.sum(dim=(-2,-1)) + gt.sum(dim=(-2,-1))  # [B]
        dice         = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
        return dice.mean()

    def forward(self, pred: torch.Tensor, gt_probs: torch.Tensor) -> dict:
        p       = pred[:, 0]                                   # [B, H, W]
        bce_v   = self.bce(p, gt_probs)
        dice_v  = self.dice_loss(p, gt_probs)
        total   = self.lambda_bce * bce_v + self.lambda_dice * dice_v
        return {'total': total, 'prob': bce_v, 'dice': dice_v}


class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.s = self.n = 0.0
    def update(self, v, n=1): self.s += v*n; self.n += n
    @property
    def avg(self): return self.s / max(self.n, 1e-8)


print(f'Loss: BCE(λ={LAMBDA_BCE}) + Dice(λ={LAMBDA_DICE})  ✓')


Loss: BCE(λ=1.0) + Dice(λ=1.0)  ✓


## Step 5 — 训练循环

**v6 变更：**
- `train_one_epoch` / `validate` 同时追踪 `prob`(BCE) 和 `dice` 两个分项
- `train_log.csv` 新增 `tr_dice` / `val_dice` 列
- 打印格式包含双损失信息


In [ ]:
# from torch.cuda.amp import GradScaler
# import json
# from datetime import datetime
# import torch # Add this line to import torch


# def set_seed(seed=42):
#     random.seed(seed); np.random.seed(seed)
#     torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
#     """
#     训练一个 epoch，追踪 total / prob(BCE) / dice 三个损失。
#     返回 dict: {total, prob, dice} 均为该 epoch 均值。
#     """
#     model.train()
#     # ★ v6：meter 增加 'dice' 分项
#     m = {k: AverageMeter() for k in ['total', 'prob', 'dice']}
#     t0, log_iv = time.time(), max(1, len(loader) // 5)

#     for step, batch in enumerate(loader):
#         phase  = batch['phase'].to(device, non_blocking=True)
#         intens = batch['intensity'].to(device, non_blocking=True)
#         # GT 直接转二值概率图，无需 labels_to_flows（v3 起删除流场分支）
#         gt_probs = torch.from_numpy(
#             gt_to_prob(batch['mask'].numpy())).to(device)        # [B, H, W]

#         optimizer.zero_grad(set_to_none=True)
#         with torch.amp.autocast('cuda'):
#             out, _ = model(phase, intens)                         # [B, 1, H, W]
#             ld     = criterion(out, gt_probs)                     # dict

#         scaler.scale(ld['total']).backward()
#         scaler.unscale_(optimizer)
#         nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         scaler.step(optimizer); scaler.update()

#         B = phase.size(0)
#         for k in ['total', 'prob', 'dice']:
#             m[k].update(ld[k].item(), B)

#         if (step + 1) % log_iv == 0:
#             print(f'  Ep{epoch:>3}[{step+1:>4}/{len(loader)}]' \
#                   f'  loss={m["total"].avg:.4f}' \
#                   f'  bce={m["prob"].avg:.4f}' \
#                   f'  dice={m["dice"].avg:.4f}' \
#                   f'  ({time.time()-t0:.0f}s)')
#     return {k: m[k].avg for k in m}


# @torch.no_grad()
# def validate(model, loader, criterion, device):
#     """
#     验证集推理，追踪 total / prob(BCE) / dice。
#     """
#     model.eval()
#     m = {k: AverageMeter() for k in ['total', 'prob', 'dice']}
#     for batch in loader:
#         phase    = batch['phase'].to(device, non_blocking=True)
#         intens   = batch['intensity'].to(device, non_blocking=True)
#         gt_probs = torch.from_numpy(
#             gt_to_prob(batch['mask'].numpy())).to(device)
#         out, _   = model(phase, intens)
#         ld       = criterion(out, gt_probs)
#         B = phase.size(0)
#         for k in ['total', 'prob', 'dice']:
#             m[k].update(ld[k].item(), B)
#     return {k: m[k].avg for k in m}


# print('Train / validate functions defined ✓')

# # ── ★ 启动训练 ─────────────────────────────────────────────────────


# set_seed(SEED)
# # ★ v6：使用 BCE+Dice 联合损失
# criterion = CellposeLoss(lambda_bce=LAMBDA_BCE, lambda_dice=LAMBDA_DICE)
# scaler    = GradScaler()

# if WARMUP_FREEZE:
#     model.freeze_backbone()

# optimizer = torch.optim.AdamW(
#     model.get_param_groups(LR_BACKBONE, LR_FUSION),
#     weight_decay=WEIGHT_DECAY)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

# best_loss, no_improve, start_epoch = float('inf'), 0, 1

# import json, glob as _glob
# from datetime import datetime

# # ── 确定 exp_id ──────────────────────────────────────────────────
# # 优先级：RESUME_EXP_ID > RESUME自动检测 > 新建（含 FINETUNE）
# if RESUME_EXP_ID.strip():
#     # 模式 A：断点续训，exp_id / 目录 / epoch 全部保持不变
#     exp_id = RESUME_EXP_ID.strip()
#     print(f'  [RESUME] Using manually specified exp_id: {exp_id}')
# elif RESUME:
#     _candidates = sorted(
#         _glob.glob(os.path.join(CKPT_DIR, '*', 'latest.pth')),
#         key=os.path.getmtime, reverse=True)
#     if _candidates:
#         try:
#             _ck_probe = torch.load(_candidates[0], map_location='cpu')
#             exp_id    = (_ck_probe.get('exp_id', '')
#                          or os.path.basename(os.path.dirname(_candidates[0])))
#             print(f'  [RESUME] Auto-detected exp: {exp_id}')
#         except Exception as e:
#             print(f'  [RESUME WARNING] Failed to load checkpoint {_candidates[0]}: {e}')
#             print('  [RESUME WARNING] Starting a new experiment instead.')
#             RESUME = False
#             exp_id = (datetime.now().strftime('%Y%m%d_%H%M%S')
#                       + (f'_{EXP_NAME}' if EXP_NAME else ''))
#     else:
#         RESUME = False
#         exp_id = (datetime.now().strftime('%Y%m%d_%H%M%S')
#                   + (f'_{EXP_NAME}' if EXP_NAME else ''))
#         print(f'  [RESUME] No checkpoint found, new exp: {exp_id}')
# else:
#     # 模式 B：新实验（含 FINETUNE_FROM_EXP_ID 权重迁移）
#     exp_id = (datetime.now().strftime('%Y%m%d_%H%M%S')
#               + (f'_{EXP_NAME}' if EXP_NAME else ''))
#     if FINETUNE_FROM_EXP_ID.strip():
#         print(f'  [FINETUNE] New exp={exp_id}, weights from {FINETUNE_FROM_EXP_ID}')
#     else:
#         print(f'  [NEW] exp_id={exp_id}')

# # ── 建立三个隔离目录 ─────────────────────────────────────────────
# exp_dir      = os.path.join(LOG_DIR,    exp_id)   # 日志
# exp_ckpt_dir = os.path.join(CKPT_DIR,  exp_id)    # 权重
# exp_out_dir  = os.path.join(OUTPUT_DIR, exp_id)   # 输出（eval/ & inference/ 在此下面）
# for _d in [exp_dir, exp_ckpt_dir, exp_out_dir]:
#     os.makedirs(_d, exist_ok=True)
# print(f'  Log   : {exp_dir}')
# print(f'  Ckpt  : {exp_ckpt_dir}')
# print(f'  Output: {exp_out_dir}')


# # ── 权重加载 ─────────────────────────────────────────────────────
# # resume_ckpt : 断点续训加载路径（latest.pth，含完整 optimizer/scheduler 状态）
# # latest_ckpt : 每 epoch 覆盖保存路径（latest.pth）
# # ★ Bug Fix（v9）: 原代码加载 best.pth 续训，导致 optimizer/scheduler 状态
# #   退回到 best 所在 epoch，而非中断时的最新 epoch。已改为 latest.pth。
# # 两者严格分离，互不干扰
# resume_ckpt = os.path.join(exp_ckpt_dir, 'latest.pth')  # ★ 修正：断点续训从 latest.pth 恢复
# latest_ckpt = os.path.join(exp_ckpt_dir, 'latest.pth')  # 保存：每 epoch 覆盖

# if RESUME and os.path.exists(resume_ckpt):
#     # ── 模式 A：断点续训（恢复 optimizer / scheduler / epoch）──
#     try:
#         ck = torch.load(resume_ckpt, map_location='cpu')
#         model.load_state_dict(ck['model'])
#         model.to(device)
#         optimizer.load_state_dict(ck['optimizer'])
#         for state in optimizer.state.values():
#             for k, v in state.items():
#                 if isinstance(v, torch.Tensor):
#                     state[k] = v.to(device)
#         scheduler.load_state_dict(ck['scheduler'])
#         start_epoch = ck['epoch'] + 1
#         best_loss   = ck.get('best_loss', float('inf'))
#         print(f'  [RESUME] start_epoch={start_epoch}  best_loss={best_loss:.6f}')
#     except Exception as e:
#         print(f'  [RESUME ERROR] {e}')
#         RESUME = False
#         start_epoch = 1
#         best_loss = float('inf')

# elif FINETUNE_FROM_EXP_ID.strip():
#     # ── 模式 B：迁移权重（只加载 model weights，optimizer/scheduler 全新）──
#     _ft_ckpt = os.path.join(CKPT_DIR, FINETUNE_FROM_EXP_ID.strip(), 'best.pth')
#     if os.path.exists(_ft_ckpt):
#         try:
#             ck = torch.load(_ft_ckpt, map_location='cpu')
#             model.load_state_dict(ck['model'])
#             model.to(device)
#             _src_epoch = ck.get('epoch', '?')
#             _src_loss  = ck.get('best_loss', float('nan'))
#             print(f'  [FINETUNE] Weights loaded from {FINETUNE_FROM_EXP_ID} '
#                   f'(src_epoch={_src_epoch}, src_best_loss={_src_loss:.6f})')
#             print(f'  [FINETUNE] optimizer & scheduler reset → new LR={LR_BACKBONE}/{LR_FUSION} takes effect')
#         except Exception as e:
#             print(f'  [FINETUNE ERROR] {e}, starting from scratch')
#     else:
#         print(f'  [FINETUNE WARNING] {_ft_ckpt} not found, starting from scratch')

# # ── 配置快照 ─────────────────────────────────────────────────────
# config_snap = {
#     'exp_id'        : exp_id,
#     'timestamp'     : datetime.now().isoformat(),
#     'start_epoch'   : start_epoch,                              # ★ 本次从哪轮开始
#     'finetune_from' : FINETUNE_FROM_EXP_ID.strip() or None,    # ★ 迁移权重来源
#     'change_note'   : CHANGE_NOTE,                              # ★ 本次改动说明
#     'BSIZE': BSIZE, 'BATCH_SIZE': BATCH_SIZE, 'EPOCHS': EPOCHS,
#     'LR_BACKBONE': LR_BACKBONE, 'LR_FUSION': LR_FUSION,
#     'LAMBDA_BCE': LAMBDA_BCE, 'LAMBDA_DICE': LAMBDA_DICE,
#     'WEIGHT_DECAY': WEIGHT_DECAY, 'VAL_RATIO': VAL_RATIO,
#     'PATIENCE': PATIENCE, 'WARMUP_EPOCHS': WARMUP_EPOCHS,
#     'PROB_THRESHOLD': PROB_THRESHOLD,
#     'n_train': len(train_samples), 'n_val': len(val_samples), 'n_test': len(test_samples),
#     'train_stems'  : [s['stem']   for s in train_samples],
#     'val_stems'    : [s['stem']   for s in val_samples],
#     'train_subdirs': sorted(set(s['subdir'] for s in train_samples)),
# }
# config_path = os.path.join(exp_dir, 'config.json')
# if os.path.exists(config_path):
#     with open(config_path) as f:
#         existing = json.load(f)
#     if not isinstance(existing, list): existing = [existing]
#     existing.append(config_snap); config_snap = existing
# with open(config_path, 'w') as f:
#     json.dump(config_snap, f, ensure_ascii=False, indent=2)

# # ── 数据集统计快照（仅新建实验时计算）────────────────────────────
# def _compute_split_stats(samples, _split_name):
#     if not samples: return {'n_samples': 0}
#     cell_counts, agg_count = [], 0
#     for s in samples:
#         mk = read_mask(s['mask'])
#         if mk is not None and mk.max() > 0:
#             _, n = ndimage.label((mk > 0).astype(np.uint8))
#             cell_counts.append(n)
#             if (mk == 2000).any(): agg_count += 1
#     cc = np.array(cell_counts) if cell_counts else np.array([0])
#     return {'n_samples': len(samples),
#             'cells_per_image': {'mean': round(float(cc.mean()),2),
#                                 'std' : round(float(cc.std()),2),
#                                 'min' : int(cc.min()), 'max': int(cc.max())},
#             'aggregate_ratio' : round(agg_count / max(len(samples),1), 4),
#             'empty_mask_ratio': round(sum(1 for c in cell_counts if c==0)
#                                       / max(len(cell_counts),1), 4)}

# _ds_stats_path = os.path.join(exp_dir, 'dataset_stats.json')
# if not os.path.exists(_ds_stats_path):
#     print('  Computing dataset stats ...')
#     _ds_stats = {'train': _compute_split_stats(train_samples, 'train'),
#                  'val'  : _compute_split_stats(val_samples,   'val'),
#                  'test' : _compute_split_stats(test_samples,  'test'),
#                  'split_seed': SEED, 'val_ratio': VAL_RATIO}
#     with open(_ds_stats_path, 'w') as _f:
#         json.dump(_ds_stats, _f, ensure_ascii=False, indent=2)
#     print(f'  dataset_stats.json → {exp_dir}')

# # ── 日志 CSV（v6：增加 tr_dice / val_dice 列）────────────────────
# log_csv = os.path.join(exp_dir, 'train_log.csv')
# if not os.path.exists(log_csv):
#     with open(log_csv, 'w', newline='') as f:
#         # ★ v6：新增 tr_dice / val_dice 列
#         csv.writer(f).writerow(['epoch','tr_prob','tr_dice','val_prob','val_dice','lr'])

# print(f'\n{"="*70}')
# print(f'  Training: epoch {start_epoch} → {EPOCHS}')
# print(f'  warmup_freeze={WARMUP_FREEZE} ({WARMUP_EPOCHS} ep)')
# print(f'  loss = {LAMBDA_BCE}×BCE + {LAMBDA_DICE}×Dice')
# print(f'  train={len(train_loader)} | val={len(val_loader)} batches')
# print(f'{"="*70}')

# try:
#     for epoch in range(start_epoch, EPOCHS + 1):

#         if WARMUP_FREEZE and epoch == WARMUP_EPOCHS + 1:
#             import gc; gc.collect(); torch.cuda.empty_cache()
#             model.unfreeze_backbone()
#             optimizer = torch.optim.AdamW(
#                 model.get_param_groups(LR_BACKBONE, LR_FUSION), weight_decay=WEIGHT_DECAY)
#             scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#                 optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)
#             print(f'  ▶ Ep{epoch}: Backbone unfrozen → full fine-tuning.')

#         tr = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch)
#         vl = validate(model, val_loader, criterion, device)
#         scheduler.step(vl['total'])
#         lr = optimizer.param_groups[0]['lr']

#         # ★ v6：打印格式包含 BCE 和 Dice 双分项
#         print(f'Ep {epoch:>3d} | '
#               f'tr_bce={tr["prob"]:.4f}  tr_dice={tr["dice"]:.4f} | '
#               f'val_bce={vl["prob"]:.4f}  val_dice={vl["dice"]:.4f} | '
#               f'lr={lr:.2e}')

#         # ★ v6：CSV 记录 tr_dice / val_dice
#         with open(log_csv, 'a', newline='') as f:
#             csv.writer(f).writerow([epoch,
#                                     tr['prob'], tr['dice'],
#                                     vl['prob'], vl['dice'],
#                                     lr])

#         # ── Checkpoint 保存（Bug #5 修复：先更新 best_loss，再构建 ck）──
#         is_best = vl['total'] < best_loss
#         if is_best:
#             best_loss  = vl['total']
#             no_improve = 0
#         else:
#             no_improve += 1

#         ck = {'epoch': epoch, 'model': model.state_dict(),
#               'optimizer': optimizer.state_dict(),
#               'scheduler': scheduler.state_dict(),
#               'best_loss': best_loss,
#               'exp_id':    exp_id}

#         # 每 epoch 覆盖 latest.pth（断点续训入口），与 best.pth 完全分离
#         torch.save(ck, latest_ckpt)
#         print(f"  [SAVE] latest.pth updated (epoch={epoch})")

#         if is_best:
#             torch.save(ck, os.path.join(exp_ckpt_dir, 'best.pth'))
#             print(f'  ★ best.pth saved  val_loss={best_loss:.6f}')

#         if WARMUP_FREEZE and epoch == WARMUP_EPOCHS:
#             _wp = os.path.join(exp_ckpt_dir, f'epoch_{epoch:03d}_warmup_end.pth')
#             torch.save(ck, _wp)
#             print(f'  📌 warmup_end checkpoint → {os.path.basename(_wp)}')

#         if no_improve >= PATIENCE:
#             print(f'\n[EarlyStopping] epoch {epoch}'); break

# except KeyboardInterrupt:
#     print('\n⚠️ 手动中断！正在保存 interrupt.pth ...')
#     ck = {'epoch': epoch, 'model': model.state_dict(),
#           'optimizer': optimizer.state_dict(),
#           'scheduler': scheduler.state_dict(),
#           'best_loss': best_loss, 'exp_id': exp_id}
#     torch.save(ck, os.path.join(exp_ckpt_dir, 'interrupt.pth'))
#     print(f'✅ 已保存 → {exp_ckpt_dir}/interrupt.pth (epoch={epoch})')

# print(f'\n✅ Training complete!  best_val_loss = {best_loss:.6f}')

# # ── 训练结束摘要（供 Step 7 写入全局注册表）─────────────────────
# _train_summary = {
#     'exp_id': exp_id, 'best_val_loss': best_loss,
#     'epochs_trained': epoch if 'epoch' in dir() else EPOCHS,
#     'timestamp_train_end': datetime.now().isoformat(),
#     'note': EXP_NAME,
#     'lambda_bce': LAMBDA_BCE, 'lambda_dice': LAMBDA_DICE,   # ★ v6
#     'n_train': len(train_samples), 'n_val': len(val_samples), 'n_test': len(test_samples),
#     'BSIZE': BSIZE, 'LR_BACKBONE': LR_BACKBONE, 'LR_FUSION': LR_FUSION,
#     'BATCH_SIZE': BATCH_SIZE, 'PATIENCE': PATIENCE,
# }
# with open(os.path.join(exp_dir, 'training_summary.json'), 'w') as _f:
#     json.dump(_train_summary, _f, ensure_ascii=False, indent=2)
# print(f'  training_summary.json → {exp_dir}')

## Step 6 — 训练曲线

v6 新增：在同一张图中绘制 BCE loss 和 Dice loss 双曲线，便于对比两个分项的收敛趋势。

**正常训练表现：**
- BCE loss 和 Dice loss 应同步下降
- Dice loss 数值通常略高于 BCE（因为 Dice 范围 [0,1] 更难压缩）
- 如果 Dice loss 居高不下而 BCE 已收敛，可尝试增大 LAMBDA_DICE


In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import glob
# from sklearn.metrics import (precision_recall_curve, roc_curve,
#                               average_precision_score, roc_auc_score)

# # 自动找最新实验目录（若 Step 5 已运行则直接使用 exp_dir）
# if 'exp_dir' not in dir() or not os.path.isdir(exp_dir):
#     exp_dirs = [d for d in sorted(glob.glob(os.path.join(LOG_DIR, '*')))
#                 if os.path.isdir(d)]
#     exp_dir  = exp_dirs[-1] if exp_dirs else LOG_DIR
# log_csv = os.path.join(exp_dir, 'train_log.csv')

# # ── Loss 曲线 ────────────────────────────────────────────────────
# if os.path.exists(log_csv):
#     df = pd.read_csv(log_csv)
#     has_dice = 'tr_dice' in df.columns

#     fig, axes = plt.subplots(1, 2 if has_dice else 1,
#                              figsize=(14 if has_dice else 10, 5))
#     if not has_dice:
#         axes = [axes, None]

#     axes[0].plot(df['epoch'], df['tr_prob'],  label='Train BCE', color='steelblue', lw=2)
#     axes[0].plot(df['epoch'], df['val_prob'], label='Val BCE',   color='tomato',    lw=2)
#     axes[0].set(xlabel='Epoch', ylabel='BCE Loss', title='Cell Probability BCE Loss')
#     axes[0].legend(); axes[0].grid(True, alpha=0.3)

#     if has_dice:
        axes[0].axvline(x=BEST_EPOCH, color='gray', linestyle='--', alpha=0.7, label=f'Best Epoch ({BEST_EPOCH})')
        axes[0].set(xlabel='Epoch', ylabel='BCE Loss', title='Cell Probability BCE Loss')
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        if has_dice:
            axes[1].plot(df['epoch'], df['tr_dice'],  label='Train Dice', color='mediumseagreen', lw=2)
            axes[1].plot(df['epoch'], df['val_dice'], label='Val Dice',   color='darkorange',    lw=2)
            # 在第 36 轮画竖虚线
            axes[1].axvline(x=BEST_EPOCH, color='gray', linestyle='--', alpha=0.7, label=f'Best Epoch ({BEST_EPOCH})')
            axes[1].set(xlabel='Epoch', ylabel='Dice Loss', title='Cell Probability Dice Loss')
            axes[1].legend(); axes[1].grid(True, alpha=0.3)

#     plt.suptitle(f'Training Curves — {os.path.basename(exp_dir)}', fontsize=11)
#     plt.tight_layout()
#     _curve_path = os.path.join(exp_dir, 'training_curves.png')
#     plt.savefig(_curve_path, bbox_inches='tight', dpi=300)
#     plt.show()
#     print(f'  training_curves.png → {exp_dir}')
# else:
#     print('No log CSV found. Run Step 5 first.')

# # ── PR / ROC 曲线（基于验证集，训练结束后一次性计算）────────────

# @torch.no_grad()
# def collect_val_probs(eval_model, loader, device):
#     """收集验证集全部像素的 sigmoid 概率和 GT 标签"""
#     eval_model.eval()
#     all_probs, all_gt = [], []
#     for batch in loader:
#         ph  = batch['phase'].to(device)
#         it  = batch['intensity'].to(device)
#         gt  = gt_to_prob(batch['mask'].numpy()).ravel()        # [B*H*W] float32
#         out, _ = eval_model(ph, it)
#         probs  = torch.sigmoid(out[:, 0]).float().cpu().numpy().ravel()
#         all_probs.append(probs)
#         all_gt.append(gt)
#     return np.concatenate(all_probs), np.concatenate(all_gt)

# # 加载 best.pth（若 Step 5 已运行且 model 在内存则复用）
# _best_pth = os.path.join(exp_ckpt_dir, 'best.pth')
# if os.path.exists(_best_pth):
#     curve_model = DualBranchTransformer(pretrained_path=None,
#                                         dtype=torch.float32).to(device)
#     curve_model.load_state_dict(
#         torch.load(_best_pth, map_location=device)['model'])
#     print(f'  [PR/ROC] Loaded {_best_pth}')
# elif 'model' in dir():
#     curve_model = model
#     print('  [PR/ROC] Using current model (best.pth not found)')
# else:
#     curve_model = None
#     print('  [PR/ROC] No model available, skipping PR/ROC.')

# if curve_model is not None:
#     probs_all, gt_all = collect_val_probs(curve_model, val_loader, device)

#     prec, rec, thr_pr = precision_recall_curve(gt_all, probs_all)
#     fpr,  tpr, _      = roc_curve(gt_all, probs_all)
#     ap_score          = average_precision_score(gt_all, probs_all)
#     auc_score         = roc_auc_score(gt_all, probs_all)

#     # 标出当前 PROB_THRESHOLD 对应的操作点
#     _idx = np.argmin(np.abs(thr_pr - PROB_THRESHOLD))

#     fig2, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(12, 5))

#     ax_pr.plot(rec, prec, color='steelblue', lw=2,
#                label=f'AP = {ap_score:.3f}')
#     ax_pr.scatter(rec[_idx], prec[_idx], s=80, color='tomato', zorder=5,
#                   label=f'thr = {PROB_THRESHOLD}  '
#                         f'P={prec[_idx]:.3f} R={rec[_idx]:.3f}')
#     ax_pr.set(xlabel='Recall', ylabel='Precision',
#               title='PR Curve (val, pixel-level)')
#     ax_pr.legend(); ax_pr.grid(True, alpha=0.3)

#     ax_roc.plot(fpr, tpr, color='mediumseagreen', lw=2,
#                 label=f'AUC = {auc_score:.3f}')
#     ax_roc.plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Random')
#     ax_roc.set(xlabel='FPR', ylabel='TPR',
#                title='ROC Curve (val, pixel-level)')
#     ax_roc.legend(); ax_roc.grid(True, alpha=0.3)

#     plt.suptitle(f'PR & ROC — {os.path.basename(exp_dir)}', fontsize=11)
#     plt.tight_layout()
#     _pr_roc_path = os.path.join(exp_dir, 'pr_roc_curves.png')
#     plt.savefig(_pr_roc_path, bbox_inches='tight', dpi=200)
#     plt.show()
#     print(f'  pr_roc_curves.png → {exp_dir}')
#     print(f'  AP (sklearn pixel) = {ap_score:.4f}   AUC = {auc_score:.4f}')

## Step 7 — 测试集评估

**v9 变更：**
- `pred_to_mask()` 删除面积/边缘阈值过滤，仅保留 sigmoid + 形态学 + 连通域标记
- AP@0.5 / AP@0.75 / AP@0.9 完整计算并保存到 test_results.csv

**eval/ 目录下文件：**
```
eval/
├── test_results.csv              逐样本 AP@0.5/0.75/0.9 + 计数偏差指标
├── test_results_stratified.csv   按药物/时间点分层统计
├── failure_analysis.csv          AP@0.5 < 0.3 失败案例
└── <subdir>/
    └── <stem>_pred_mask.tiff     最终实例 mask uint16
```


In [ ]:
from cellpose import metrics as cp_metrics
import pandas as pd


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -80, 80)))


def pred_to_mask(pred_np: np.ndarray) -> np.ndarray:
    """
    将模型输出转换为实例分割 mask（v9：sigmoid + 形态学，无阈值过滤）。

    参数:
        pred_np : [1, H, W]  cellprob logit

    返回:
        final_mask : [H, W] uint32  实例编号 mask（0=背景，≥1 为各独立实例）

    流程:
        1. sigmoid(logit) → 概率图 [H,W]
        2. > PROB_THRESHOLD → binary 二值图
        3. binary_opening(iter=1) + binary_dilation(iter=5) + binary_erosion(iter=5) → 形态学清洁
           ★ Fix1: 原 iter=2/8/8 过激，会切断细长/非凸细胞的窄连接 → 一个细胞碎成多个实例
           ★ 放宽后保留去噪能力的同时不破坏连通性
        4. ndimage.label(binary) → final_mask（直接返回，无面积/边缘过滤）
    """
    H, W = pred_np.shape[1], pred_np.shape[2]

    # ── 1. 概率图 ────────────────────────────────────────────────
    cellprob = sigmoid(pred_np[0]).reshape(H, W).astype(np.float32)

    # ── 2. 二值化 ─────────────────────────────────────────────────
    binary = (cellprob > PROB_THRESHOLD).astype(np.uint8)

    # ── 3. 形态学清洁（去毛刺 / 填洞 / 平滑边界）─────────────────
    # ★ Fix1: opening iter 2→1，dilation/erosion iter 8→5
    #   原参数对细长或非凸血小板过度腐蚀，导致一个连通体被切成多个实例
    binary = ndimage.binary_opening(binary,  iterations=1).astype(np.uint8)
    binary = ndimage.binary_dilation(binary, iterations=5).astype(np.uint8)
    binary = ndimage.binary_erosion(binary,  iterations=5).astype(np.uint8)

    # ── 4. 连通域标记 → 直接返回实例 mask ────────────────────────
    final_mask, _ = ndimage.label(binary)
    return final_mask.astype(np.uint32)


# ── Step 7 独立运行所需 import（跳过 Step 4/5 时补全）──────────
import json, glob as _glob
from datetime import datetime
from scipy import ndimage  # pred_to_mask / GT label 均依赖

# ── 指定评估哪个实验（可选）────────────────────────────────────
# 留空 → 自动使用 Step 5 产生的 exp_id，或扫描最新 best.pth
# 填写 → 强制评估该实验，适合跳过训练直接评估
#EVAL_EXP_ID = ''   # 例如 '20260312_110103'

# ── 加载 best 模型 ──────────────────────────────────────────────
# Step 7 可独立运行（不依赖 Step 5 变量）：自动从最新 checkpoint 推断 exp_id
if EVAL_EXP_ID.strip():
    # 模式 1：手动指定评估实验
    exp_id       = EVAL_EXP_ID.strip()
    exp_ckpt_dir = os.path.join(CKPT_DIR,   exp_id)
    exp_dir      = os.path.join(LOG_DIR,     exp_id)
    exp_out_dir  = os.path.join(OUTPUT_DIR,  exp_id)
    os.makedirs(exp_out_dir, exist_ok=True)
    print(f'  [EVAL] Using specified exp_id: {exp_id}')
elif 'exp_ckpt_dir' not in dir():
    # 模式 2：Step 5 未运行，自动扫描最新 best.pth
    _cands = sorted(_glob.glob(os.path.join(CKPT_DIR, '*', 'best.pth')),
                    key=os.path.getmtime, reverse=True)
    if _cands:
        _ck_probe    = torch.load(_cands[0], map_location='cpu')
        exp_id       = _ck_probe.get('exp_id', os.path.basename(os.path.dirname(_cands[0])))
        exp_ckpt_dir = os.path.dirname(_cands[0])
    else:
        exp_id       = 'unknown_exp'
        exp_ckpt_dir = CKPT_DIR
    exp_dir     = os.path.join(LOG_DIR,    exp_id)
    exp_out_dir = os.path.join(OUTPUT_DIR, exp_id)
    os.makedirs(exp_out_dir, exist_ok=True)
    print(f'  [EVAL] Auto-detected exp_id: {exp_id}')
else:
    # 模式 3：Step 5 已运行，直接使用其 exp_id
    print(f'  [EVAL] Using exp_id from Step 5: {exp_id}')

# ★ v6：评估结果保存到 eval/ 子目录，与 inference/ 分离
exp_eval_dir = os.path.join(exp_out_dir, 'eval')
os.makedirs(exp_eval_dir, exist_ok=True)
print(f'  Eval output dir: {exp_eval_dir}')

best_ckpt = os.path.join(exp_ckpt_dir, 'best.pth')
if os.path.exists(best_ckpt):
    eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
    ck = torch.load(best_ckpt, map_location=device)
    eval_model.load_state_dict(ck['model'])
    print(f'Loaded {best_ckpt}')
    print(f'  epoch={ck["epoch"]}  val_loss={ck["best_loss"]:.6f}')
else:
    print('[INFO] best.pth not found, using current model.')
    eval_model = model

eval_model.eval()

ap50_list, results = [], []

for i, s in enumerate(test_samples):
    ph_arr = read_rgb(s['phase'])
    it_arr = read_rgb(s['intensity'])
    mk_arr = read_mask(s['mask'])
    stem   = s['stem']

    # 滑动窗口推理 → 完整原图尺寸的 logit
    logit_full, ph_full, it_full = sliding_window_inference(eval_model, ph_arr, it_arr)

    # v8：pred_to_mask 直接返回 final_mask（无图合并）
    pred_mask = pred_to_mask(logit_full)

    # ── ★ 保存到 eval/<subdir>/ ──────────────────────────────────
    import tifffile
    _save_dir = os.path.join(exp_eval_dir, s['subdir'])
    os.makedirs(_save_dir, exist_ok=True)
    tifffile.imwrite(os.path.join(_save_dir, f'{stem}_pred_mask.tiff'),
                     pred_mask.astype(np.uint16))

    if mk_arr is not None and mk_arr.max() > 0:
        gt_instance, n_gt = ndimage.label((mk_arr > 0).astype(np.uint8))

        ap, _, _, _ = cp_metrics.average_precision(
            [gt_instance.astype(np.uint16)],
            [pred_mask.astype(np.uint16)],
            threshold=[0.5, 0.75, 0.9],
        )

        n_pred = int(pred_mask.max())
        ap50_list.append(float(ap[0][0]))
        _bias  = n_pred - n_gt

        results.append({
            'stem'       : stem,
            'subdir'     : s['subdir'],
            'AP@0.5'     : round(float(ap[0][0]), 4),
            'AP@0.75'    : round(float(ap[0][1]), 4),
            'AP@0.9'     : round(float(ap[0][2]), 4),
            'pred_cells' : n_pred,
            'gt_cells'   : n_gt,
            'count_bias' : _bias,
            'count_mae'  : abs(_bias),
            'count_mape' : round(abs(_bias) / max(n_gt, 1), 4),
        })

    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/{len(test_samples)}]  AP@0.5={np.mean(ap50_list):.4f}')

print(f'\n{"="*55}')
print(f'  Test Results ({len(results)} images with GT)')
print(f'{"="*55}')
if ap50_list:
    print(f'  AP@0.5  : {np.mean(ap50_list):.4f} ± {np.std(ap50_list):.4f}')

if results:
    df_res = pd.DataFrame(results)

    # ── 逐样本结果 ───────────────────────────────────────────────
    csv_out = os.path.join(exp_eval_dir, 'test_results.csv')
    df_res.to_csv(csv_out, index=False)
    print(f'  test_results.csv → {csv_out}')
    print(df_res[['AP@0.5','AP@0.75','AP@0.9',
                  'pred_cells','gt_cells',
                  'count_bias','count_mae']].describe().round(4))

    # ── 汇总统计 CSV（每次评估一行）──────────────────────────────
    _sum_row = {
        'exp_id'          : exp_id,
        'timestamp'       : datetime.now().isoformat(),
        'n_samples'       : len(df_res),
        'AP@0.5_mean'     : round(float(df_res['AP@0.5'].mean()),     4),
        'AP@0.5_std'      : round(float(df_res['AP@0.5'].std()),      4),
        'AP@0.5_median'   : round(float(df_res['AP@0.5'].median()),   4),
        'AP@0.75_mean'    : round(float(df_res['AP@0.75'].mean()),    4),
        'AP@0.9_mean'     : round(float(df_res['AP@0.9'].mean()),     4),
        'pred_cells_mean' : round(float(df_res['pred_cells'].mean()), 2),
        'pred_cells_std'  : round(float(df_res['pred_cells'].std()),  2),
        'count_bias_mean' : round(float(df_res['count_bias'].mean()), 3),
        'count_mae_mean'  : round(float(df_res['count_mae'].mean()),  3),
        'count_mape_mean' : round(float(df_res['count_mape'].mean()), 4),
        'gt_cells_mean'   : round(float(df_res['gt_cells'].mean()),   2),
        'gt_cells_std'    : round(float(df_res['gt_cells'].std()),    2),
        'LR_BACKBONE'     : LR_BACKBONE,
        'LR_FUSION'       : LR_FUSION,
        'LAMBDA_BCE'      : LAMBDA_BCE,
        'LAMBDA_DICE'     : LAMBDA_DICE,
        'finetune_from'   : FINETUNE_FROM_EXP_ID.strip() or '',
        'change_note'     : CHANGE_NOTE,
    }
    _sum_path = os.path.join(LOG_DIR, 'test_results_summary.csv')
    _sum_exists = os.path.exists(_sum_path)
    import csv as _csv_sum
    with open(_sum_path, 'a', newline='', encoding='utf-8') as _sf:
        _sw = _csv_sum.DictWriter(_sf, fieldnames=list(_sum_row.keys()))
        if not _sum_exists: _sw.writeheader()
        _sw.writerow(_sum_row)
    print(f'  test_results_summary.csv (汇总行) → {_sum_path}')

    # 分层评估（按 subdir）
    _strat = (df_res.groupby('subdir')
              .agg(n_samples      =('stem',        'count'),
                   AP05_mean      =('AP@0.5',      'mean'),
                   AP05_std       =('AP@0.5',      'std'),
                   AP075_mean     =('AP@0.75',     'mean'),
                   count_bias_mean=('count_bias',  'mean'),
                   gt_mean        =('gt_cells',    'mean'),
                   pred_mean      =('pred_cells',  'mean'))
              .round(4).reset_index())
    _strat['risk_flag'] = _strat['count_bias_mean'].abs().apply(
        lambda x: '⚠️ HIGH_BIAS' if x > 10 else '')
    _strat_path = os.path.join(exp_eval_dir, 'test_results_stratified.csv')
    _strat.to_csv(_strat_path, index=False)
    print(f'  test_results_stratified.csv → {_strat_path}')
    print(_strat[['subdir','n_samples','AP05_mean','count_bias_mean','risk_flag']].to_string())

    # 失败案例日志
    _fail = df_res[df_res['AP@0.5'] < 0.3].copy()
    if len(_fail):
        def _classify_failure(row):
            if row['pred_cells'] == 0:              return 'complete_miss'
            ratio = row['pred_cells'] / max(row['gt_cells'], 1)
            if ratio < 0.5:                         return 'severe_undercount'
            if ratio > 2.0:                         return 'severe_overcount'
            return 'low_iou'
        _fail['failure_type'] = _fail.apply(_classify_failure, axis=1)
        _fail_path = os.path.join(exp_eval_dir, 'failure_analysis.csv')
        _fail.to_csv(_fail_path, index=False)
        print(f'\n  ⚠️  {len(_fail)} failures (AP@0.5<0.3) → {_fail_path}')
        print(_fail[['stem','AP@0.5','gt_cells','pred_cells','failure_type']].to_string())
    else:
        print('\n  ✅ No failure samples (AP@0.5 < 0.3)')

    # 全局实验注册表
    _reg_path = os.path.join(LOG_DIR, 'experiments_registry.csv')
    _reg_cols = ['exp_id','timestamp','best_val_loss',
                 'AP@0.5_mean','AP@0.5_std','AP@0.75_mean',
                 'count_bias_mean','count_mae_mean',
                 'epochs_trained','n_train','n_val','n_test',
                 'BSIZE','LR_BACKBONE','LR_FUSION','LAMBDA_BCE','LAMBDA_DICE',  # ★ v6
                 'BATCH_SIZE','PATIENCE','note']
    _ts_path = os.path.join(exp_dir, 'training_summary.json')
    _ts = json.load(open(_ts_path)) if os.path.exists(_ts_path) else {}
    _reg_row = {
        'exp_id'          : exp_id,
        'timestamp'       : datetime.now().isoformat(),
        'best_val_loss'   : round(_ts.get('best_val_loss', float('nan')), 6),
        'AP@0.5_mean'     : round(float(np.mean(ap50_list)), 4),
        'AP@0.5_std'      : round(float(np.std(ap50_list)),  4),
        'AP@0.75_mean'    : round(float(df_res['AP@0.75'].mean()), 4),
        'count_bias_mean' : round(float(df_res['count_bias'].mean()), 3),
        'count_mae_mean'  : round(float(df_res['count_mae'].mean()),  3),
        'epochs_trained'  : _ts.get('epochs_trained', ''),
        'n_train'         : _ts.get('n_train', ''),
        'n_val'           : _ts.get('n_val',   ''),
        'n_test'          : len(test_samples),
        'BSIZE'           : BSIZE,
        'LR_BACKBONE'     : LR_BACKBONE,
        'LR_FUSION'       : LR_FUSION,
        'LAMBDA_BCE'      : _ts.get('lambda_bce',  LAMBDA_BCE),   # ★ v6
        'LAMBDA_DICE'     : _ts.get('lambda_dice', LAMBDA_DICE),  # ★ v6
        'BATCH_SIZE'      : BATCH_SIZE,
        'PATIENCE'        : PATIENCE,
        'note'            : _ts.get('note', EXP_NAME),
    }
    _reg_exists = os.path.exists(_reg_path)
    import csv as _csv_mod
    with open(_reg_path, 'a', newline='') as _rf:
        _w = _csv_mod.DictWriter(_rf, fieldnames=_reg_cols)
        if not _reg_exists: _w.writeheader()
        _w.writerow(_reg_row)
    print(f'\n  experiments_registry.csv updated → {_reg_path}')

print(df_res[['AP@0.5','AP@0.75','AP@0.9','count_bias','count_mae']].describe().round(4))

  [EVAL] Using specified exp_id: 20260326_103536
  Eval output dir: /content/drive/MyDrive/drug_effect/seg/outputs/20260326_103536/eval
[INFO] 未指定权重路径，随机初始化。
Loaded /content/drive/MyDrive/drug_effect/seg/ckpts/20260326_103536/best.pth
  epoch=36  val_loss=0.162288
  [20/200]  AP@0.5=0.9583
  [40/200]  AP@0.5=0.9792
  [60/200]  AP@0.5=0.9861
  [80/200]  AP@0.5=0.9896
  [100/200]  AP@0.5=0.9917
  [120/200]  AP@0.5=0.9889
  [140/200]  AP@0.5=0.9905
  [160/200]  AP@0.5=0.9917
  [180/200]  AP@0.5=0.9926
  [200/200]  AP@0.5=0.9933

  Test Results (200 images with GT)
  AP@0.5  : 0.9933 ± 0.0549
  test_results.csv → /content/drive/MyDrive/drug_effect/seg/outputs/20260326_103536/eval/test_results.csv
         AP@0.5   AP@0.75    AP@0.9  pred_cells  gt_cells  count_bias  \
count  200.0000  200.0000  200.0000     200.000  200.0000    200.0000   
mean     0.9933    0.9204    0.3900       1.050    1.0400      0.0100   
std      0.0550    0.2651    0.4855       0.385    0.2622      0.1734   
min   

In [ ]:
from cellpose import metrics as cp_metrics
import pandas as pd


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -80, 80)))


def pred_to_mask(pred_np: np.ndarray) -> np.ndarray:
    """
    将模型输出转换为实例分割 mask（v9：sigmoid + 形态学，无阈值过滤）。

    参数:
        pred_np : [1, H, W]  cellprob logit

    返回:
        final_mask : [H, W] uint32  实例编号 mask（0=背景，≥1 为各独立实例）

    流程:
        1. sigmoid(logit) → 概率图 [H,W]
        2. > PROB_THRESHOLD → binary 二值图
        3. binary_opening(iter=2) + binary_dilation(iter=8) + binary_erosion(iter=8) → 形态学清洁
        4. ndimage.label(binary) → final_mask（直接返回，无面积/边缘过滤）
    """
    H, W = pred_np.shape[1], pred_np.shape[2]

    # ── 1. 概率图 ────────────────────────────────────────────────
    cellprob = sigmoid(pred_np[0]).reshape(H, W).astype(np.float32)

    # ── 2. 二值化 ─────────────────────────────────────────────────
    binary = (cellprob > PROB_THRESHOLD).astype(np.uint8)

    # ── 3. 形态学清洁（去毛刺 / 填洞 / 平滑边界）─────────────────
    binary = ndimage.binary_opening(binary,  iterations=2).astype(np.uint8)
    binary = ndimage.binary_dilation(binary, iterations=8).astype(np.uint8)
    binary = ndimage.binary_erosion(binary,  iterations=8).astype(np.uint8)

    # ── 4. 连通域标记 → 直接返回实例 mask ────────────────────────
    final_mask, _ = ndimage.label(binary)
    return final_mask.astype(np.uint32)


# # ── Step 7 独立运行所需 import（跳过 Step 4/5 时补全）──────────
# import json, glob as _glob
# from datetime import datetime
# from scipy import ndimage  # pred_to_mask / GT label 均依赖

# # ── 指定评估哪个实验（可选）────────────────────────────────────
# # 留空 → 自动使用 Step 5 产生的 exp_id，或扫描最新 best.pth
# # 填写 → 强制评估该实验，适合跳过训练直接评估
# #EVAL_EXP_ID = ''   # 例如 '20260312_110103'

# # ── 加载 best 模型 ──────────────────────────────────────────────
# # Step 7 可独立运行（不依赖 Step 5 变量）：自动从最新 checkpoint 推断 exp_id
# if EVAL_EXP_ID.strip():
#     # 模式 1：手动指定评估实验
#     exp_id       = EVAL_EXP_ID.strip()
#     exp_ckpt_dir = os.path.join(CKPT_DIR,   exp_id)
#     exp_dir      = os.path.join(LOG_DIR,     exp_id)
#     exp_out_dir  = os.path.join(OUTPUT_DIR,  exp_id)
#     os.makedirs(exp_out_dir, exist_ok=True)
#     print(f'  [EVAL] Using specified exp_id: {exp_id}')
# elif 'exp_ckpt_dir' not in dir():
#     # 模式 2：Step 5 未运行，自动扫描最新 best.pth
#     _cands = sorted(_glob.glob(os.path.join(CKPT_DIR, '*', 'best.pth')),
#                     key=os.path.getmtime, reverse=True)
#     if _cands:
#         _ck_probe    = torch.load(_cands[0], map_location='cpu')
#         exp_id       = _ck_probe.get('exp_id', os.path.basename(os.path.dirname(_cands[0])))
#         exp_ckpt_dir = os.path.dirname(_cands[0])
#     else:
#         exp_id       = 'unknown_exp'
#         exp_ckpt_dir = CKPT_DIR
#     exp_dir     = os.path.join(LOG_DIR,    exp_id)
#     exp_out_dir = os.path.join(OUTPUT_DIR, exp_id)
#     os.makedirs(exp_out_dir, exist_ok=True)
#     print(f'  [EVAL] Auto-detected exp_id: {exp_id}')
# else:
#     # 模式 3：Step 5 已运行，直接使用其 exp_id
#     print(f'  [EVAL] Using exp_id from Step 5: {exp_id}')

# # ★ v6：评估结果保存到 eval/ 子目录，与 inference/ 分离
# exp_eval_dir = os.path.join(exp_out_dir, 'eval')
# os.makedirs(exp_eval_dir, exist_ok=True)
# print(f'  Eval output dir: {exp_eval_dir}')

# best_ckpt = os.path.join(exp_ckpt_dir, 'best.pth')
# if os.path.exists(best_ckpt):
#     eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
#     ck = torch.load(best_ckpt, map_location=device)
#     eval_model.load_state_dict(ck['model'])
#     print(f'Loaded {best_ckpt}')
#     print(f'  epoch={ck["epoch"]}  val_loss={ck["best_loss"]:.6f}')
# else:
#     print('[INFO] best.pth not found, using current model.')
#     eval_model = model

# eval_model.eval()

# ap50_list, results = [], []

# for i, s in enumerate(test_samples):
#     ph_arr = read_rgb(s['phase'])
#     it_arr = read_rgb(s['intensity'])
#     mk_arr = read_mask(s['mask'])
#     stem   = s['stem']

#     # 滑动窗口推理 → 完整原图尺寸的 logit
#     logit_full, ph_full, it_full = sliding_window_inference(eval_model, ph_arr, it_arr)

#     # v8：pred_to_mask 直接返回 final_mask（无图合并）
#     pred_mask = pred_to_mask(logit_full)

#     # ── ★ 保存到 eval/<subdir>/ ──────────────────────────────────
#     import tifffile
#     _save_dir = os.path.join(exp_eval_dir, s['subdir'])
#     os.makedirs(_save_dir, exist_ok=True)
#     tifffile.imwrite(os.path.join(_save_dir, f'{stem}_pred_mask.tiff'),
#                      pred_mask.astype(np.uint16))

#     if mk_arr is not None and mk_arr.max() > 0:
#         gt_instance, n_gt = ndimage.label((mk_arr > 0).astype(np.uint8))

#         ap, _, _, _ = cp_metrics.average_precision(
#             [gt_instance.astype(np.uint16)],
#             [pred_mask.astype(np.uint16)],
#             threshold=[0.5, 0.75, 0.9],
#         )

#         n_pred = int(pred_mask.max())
#         ap50_list.append(float(ap[0][0]))
#         _bias  = n_pred - n_gt

#         results.append({
#             'stem'       : stem,
#             'subdir'     : s['subdir'],
#             'AP@0.5'     : round(float(ap[0][0]), 4),
#             'AP@0.75'    : round(float(ap[0][1]), 4),
#             'AP@0.9'     : round(float(ap[0][2]), 4),
#             'pred_cells' : n_pred,
#             'gt_cells'   : n_gt,
#             'count_bias' : _bias,
#             'count_mae'  : abs(_bias),
#             'count_mape' : round(abs(_bias) / max(n_gt, 1), 4),
#         })

#     if (i + 1) % 20 == 0:
#         print(f'  [{i+1}/{len(test_samples)}]  AP@0.5={np.mean(ap50_list):.4f}')

# print(f'\n{"="*55}')
# print(f'  Test Results ({len(results)} images with GT)')
# print(f'{"="*55}')
# if ap50_list:
#     print(f'  AP@0.5  : {np.mean(ap50_list):.4f} ± {np.std(ap50_list):.4f}')

# if results:
#     df_res = pd.DataFrame(results)

#     # ── 逐样本结果 ───────────────────────────────────────────────
#     csv_out = os.path.join(exp_eval_dir, 'test_results.csv')
#     df_res.to_csv(csv_out, index=False)
#     print(f'  test_results.csv → {csv_out}')
#     print(df_res[['AP@0.5','AP@0.75','AP@0.9',
#                   'pred_cells','gt_cells',
#                   'count_bias','count_mae']].describe().round(4))

#     # ── 汇总统计 CSV（每次评估一行）──────────────────────────────
#     _sum_row = {
#         'exp_id'          : exp_id,
#         'timestamp'       : datetime.now().isoformat(),
#         'n_samples'       : len(df_res),
#         'AP@0.5_mean'     : round(float(df_res['AP@0.5'].mean()),     4),
#         'AP@0.5_std'      : round(float(df_res['AP@0.5'].std()),      4),
#         'AP@0.5_median'   : round(float(df_res['AP@0.5'].median()),   4),
#         'AP@0.75_mean'    : round(float(df_res['AP@0.75'].mean()),    4),
#         'AP@0.9_mean'     : round(float(df_res['AP@0.9'].mean()),     4),
#         'pred_cells_mean' : round(float(df_res['pred_cells'].mean()), 2),
#         'pred_cells_std'  : round(float(df_res['pred_cells'].std()),  2),
#         'count_bias_mean' : round(float(df_res['count_bias'].mean()), 3),
#         'count_mae_mean'  : round(float(df_res['count_mae'].mean()),  3),
#         'count_mape_mean' : round(float(df_res['count_mape'].mean()), 4),
#         'gt_cells_mean'   : round(float(df_res['gt_cells'].mean()),   2),
#         'gt_cells_std'    : round(float(df_res['gt_cells'].std()),    2),
#         'LR_BACKBONE'     : LR_BACKBONE,
#         'LR_FUSION'       : LR_FUSION,
#         'LAMBDA_BCE'      : LAMBDA_BCE,
#         'LAMBDA_DICE'     : LAMBDA_DICE,
#         'finetune_from'   : FINETUNE_FROM_EXP_ID.strip() or '',
#         'change_note'     : CHANGE_NOTE,
#     }
#     _sum_path = os.path.join(LOG_DIR, 'test_results_summary.csv')
#     _sum_exists = os.path.exists(_sum_path)
#     import csv as _csv_sum
#     with open(_sum_path, 'a', newline='', encoding='utf-8') as _sf:
#         _sw = _csv_sum.DictWriter(_sf, fieldnames=list(_sum_row.keys()))
#         if not _sum_exists: _sw.writeheader()
#         _sw.writerow(_sum_row)
#     print(f'  test_results_summary.csv (汇总行) → {_sum_path}')

#     # 分层评估（按 subdir）
#     _strat = (df_res.groupby('subdir')
#               .agg(n_samples      =('stem',        'count'),
#                    AP05_mean      =('AP@0.5',      'mean'),
#                    AP05_std       =('AP@0.5',      'std'),
#                    AP075_mean     =('AP@0.75',     'mean'),
#                    count_bias_mean=('count_bias',  'mean'),
#                    gt_mean        =('gt_cells',    'mean'),
#                    pred_mean      =('pred_cells',  'mean'))
#               .round(4).reset_index())
#     _strat['risk_flag'] = _strat['count_bias_mean'].abs().apply(
#         lambda x: '⚠️ HIGH_BIAS' if x > 10 else '')
#     _strat_path = os.path.join(exp_eval_dir, 'test_results_stratified.csv')
#     _strat.to_csv(_strat_path, index=False)
#     print(f'  test_results_stratified.csv → {_strat_path}')
#     print(_strat[['subdir','n_samples','AP05_mean','count_bias_mean','risk_flag']].to_string())

#     # 失败案例日志
#     _fail = df_res[df_res['AP@0.5'] < 0.3].copy()
#     if len(_fail):
#         def _classify_failure(row):
#             if row['pred_cells'] == 0:              return 'complete_miss'
#             ratio = row['pred_cells'] / max(row['gt_cells'], 1)
#             if ratio < 0.5:                         return 'severe_undercount'
#             if ratio > 2.0:                         return 'severe_overcount'
#             return 'low_iou'
#         _fail['failure_type'] = _fail.apply(_classify_failure, axis=1)
#         _fail_path = os.path.join(exp_eval_dir, 'failure_analysis.csv')
#         _fail.to_csv(_fail_path, index=False)
#         print(f'\n  ⚠️  {len(_fail)} failures (AP@0.5<0.3) → {_fail_path}')
#         print(_fail[['stem','AP@0.5','gt_cells','pred_cells','failure_type']].to_string())
#     else:
#         print('\n  ✅ No failure samples (AP@0.5 < 0.3)')

#     # 全局实验注册表
#     _reg_path = os.path.join(LOG_DIR, 'experiments_registry.csv')
#     _reg_cols = ['exp_id','timestamp','best_val_loss',
#                  'AP@0.5_mean','AP@0.5_std','AP@0.75_mean',
#                  'count_bias_mean','count_mae_mean',
#                  'epochs_trained','n_train','n_val','n_test',
#                  'BSIZE','LR_BACKBONE','LR_FUSION','LAMBDA_BCE','LAMBDA_DICE',  # ★ v6
#                  'BATCH_SIZE','PATIENCE','note']
#     _ts_path = os.path.join(exp_dir, 'training_summary.json')
#     _ts = json.load(open(_ts_path)) if os.path.exists(_ts_path) else {}
#     _reg_row = {
#         'exp_id'          : exp_id,
#         'timestamp'       : datetime.now().isoformat(),
#         'best_val_loss'   : round(_ts.get('best_val_loss', float('nan')), 6),
#         'AP@0.5_mean'     : round(float(np.mean(ap50_list)), 4),
#         'AP@0.5_std'      : round(float(np.std(ap50_list)),  4),
#         'AP@0.75_mean'    : round(float(df_res['AP@0.75'].mean()), 4),
#         'count_bias_mean' : round(float(df_res['count_bias'].mean()), 3),
#         'count_mae_mean'  : round(float(df_res['count_mae'].mean()),  3),
#         'epochs_trained'  : _ts.get('epochs_trained', ''),
#         'n_train'         : _ts.get('n_train', ''),
#         'n_val'           : _ts.get('n_val',   ''),
#         'n_test'          : len(test_samples),
#         'BSIZE'           : BSIZE,
#         'LR_BACKBONE'     : LR_BACKBONE,
#         'LR_FUSION'       : LR_FUSION,
#         'LAMBDA_BCE'      : _ts.get('lambda_bce',  LAMBDA_BCE),   # ★ v6
#         'LAMBDA_DICE'     : _ts.get('lambda_dice', LAMBDA_DICE),  # ★ v6
#         'BATCH_SIZE'      : BATCH_SIZE,
#         'PATIENCE'        : PATIENCE,
#         'note'            : _ts.get('note', EXP_NAME),
#     }
#     _reg_exists = os.path.exists(_reg_path)
#     import csv as _csv_mod
#     with open(_reg_path, 'a', newline='') as _rf:
#         _w = _csv_mod.DictWriter(_rf, fieldnames=_reg_cols)
#         if not _reg_exists: _w.writeheader()
#         _w.writerow(_reg_row)
#     print(f'\n  experiments_registry.csv updated → {_reg_path}')

# print(df_res[['AP@0.5','AP@0.75','AP@0.9','count_bias','count_mae']].describe().round(4))


## Step 8 — 单样本可视化

**v8 对比图：5 列（有 GT）/ 4 列（无 GT）**

```
列1  Phase (RGB)      列2  Intensity (Gray)  列3  Cell Probability
列4  Predicted Masks  列5  GT Masks（有 GT 时）
```

可视化工具函数（Step 8 & Step 9 共用）：
- `draw_masks_on_gray()` ：绘制实例掩膜（随机彩色叠加）
- `visualize_sample()`   ：组合对比图（4/5 列）


In [ ]:
import pandas as pd
import glob as _glob

# ── ★ v7：指定推理哪个实验（仿照 Step 7 EVAL_EXP_ID 模式）────────
# 留空 → 自动使用 Step 7/5 产生的 exp_id，或扫描最新 best.pth
# 填写 → 强制推理该实验，适合跳过训练/评估直接批量推理
if INFER_EXP_ID.strip():
    # 模式 1：手动指定推理实验
    exp_id       = INFER_EXP_ID.strip()
    exp_ckpt_dir = os.path.join(CKPT_DIR,   exp_id)
    exp_dir      = os.path.join(LOG_DIR,     exp_id)
    exp_out_dir  = os.path.join(OUTPUT_DIR,  exp_id)
    os.makedirs(exp_out_dir, exist_ok=True)
    print(f'  [INFER] Using specified exp_id: {exp_id}')
elif 'exp_ckpt_dir' not in dir():
    # 模式 2：Step 5/7 未运行，自动扫描最新 best.pth
    _cands = sorted(_glob.glob(os.path.join(CKPT_DIR, '*', 'best.pth')),
                    key=os.path.getmtime, reverse=True)
    if _cands:
        _ck_probe    = torch.load(_cands[0], map_location='cpu')
        exp_id       = _ck_probe.get('exp_id', os.path.basename(os.path.dirname(_cands[0])))
        exp_ckpt_dir = os.path.dirname(_cands[0])
    else:
        exp_id       = 'unknown_exp'
        exp_ckpt_dir = CKPT_DIR
    exp_dir     = os.path.join(LOG_DIR,    exp_id)
    exp_out_dir = os.path.join(OUTPUT_DIR, exp_id)
    os.makedirs(exp_out_dir, exist_ok=True)
    print(f'  [INFER] Auto-detected exp_id: {exp_id}')
else:
    # 模式 3：Step 5/7 已运行，直接使用其 exp_id
    print(f'  [INFER] Using exp_id from Step 5/7: {exp_id}')

# ── ★ v7：加载推理模型 ────────────────────────────────────────────
# INFER_EXP_ID 指定时强制重新加载对应实验的 best.pth
# 否则复用 Step 7 已加载的 eval_model（避免重复加载）
_inf_ckpt = os.path.join(exp_ckpt_dir, 'best.pth')
if INFER_EXP_ID.strip() and os.path.exists(_inf_ckpt):
    eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
    _ck = torch.load(_inf_ckpt, map_location=device)
    eval_model.load_state_dict(_ck['model'])
    print(f'  Loaded {_inf_ckpt}')
    print(f'  epoch={_ck["epoch"]}  val_loss={_ck["best_loss"]:.6f}')
elif 'eval_model' not in dir():
    if os.path.exists(_inf_ckpt):
        eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
        _ck = torch.load(_inf_ckpt, map_location=device)
        eval_model.load_state_dict(_ck['model'])
        print(f'  Loaded {_inf_ckpt}')
        print(f'  epoch={_ck["epoch"]}  val_loss={_ck["best_loss"]:.6f}')
    else:
        print(f'  [INFER WARNING] best.pth not found at {_inf_ckpt}, using current model.')
        eval_model = model
else:
    print(f'  [INFER] Reusing eval_model from Step 7/5')

print(f'Batch inference on {len(test_samples)} samples ...')
eval_model.eval()
summary = []

In [ ]:
# import cv2
# import matplotlib.pyplot as plt
# import tifffile


# # ═══════════════════════════════════════════════════════════════
# # 可视化工具函数
# # ═══════════════════════════════════════════════════════════════

# def draw_masks_on_gray(gray_img: np.ndarray,
#                        mask: np.ndarray,
#                        alpha: float = 0.75) -> np.ndarray:
#     """
#     合并后实例掩膜（随机彩色填充）叠加在灰度图上。

#     Args:
#         gray_img : [H, W] uint8  灰度图
#         mask     : [H, W]        实例编号 mask（0=背景）
#         alpha    : 填充透明度
#     """
#     canvas       = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2RGB).astype(np.float32)
#     instance_ids = np.unique(mask)[1:]     # 跳过背景 0
#     np.random.seed(42)
#     colors = np.random.randint(100, 255, size=(len(instance_ids), 3))
#     for cid, col in zip(instance_ids, colors):
#         region = (mask == cid)
#         for c in range(3):
#             canvas[region, c] = alpha * col[c] + (1 - alpha) * canvas[region, c]
#     return canvas.astype(np.uint8)


# def visualize_sample(ph_c: np.ndarray,
#                      it_c: np.ndarray,
#                      cellprob: np.ndarray,
#                      pred_mask: np.ndarray,
#                      gt_instance: np.ndarray = None,
#                      stem: str = '',
#                      subdir: str = '') -> plt.Figure:
#     """
#     生成对比图（有 GT 时 5 列，无 GT 时 4 列）。

#     列1  Phase (RGB)      : 相位图原图
#     列2  Intensity (Gray) : 明场图（灰度）
#     列3  Cell Probability : cellprob 热力图（越亮越可能是细胞）
#     列4  Predicted Masks  : 阈值过滤后最终实例掩膜（随机彩色）
#     列5  GT Masks         : Ground Truth（仅在有 GT 时显示）

#     Args:
#         ph_c        : [H, W, 3] uint8  相位图
#         it_c        : [H, W, 3] uint8  明场图（RGB）
#         cellprob    : [H, W] float32   sigmoid 概率图，范围 (0,1)
#         pred_mask   : [H, W] uint32    预测实例 mask
#         gt_instance : [H, W] 可选      GT 实例 mask
#         stem        : 文件名（标题用）
#         subdir      : 子目录（标题用）
#     """
#     it_gray = it_c[..., 0].astype(np.uint8)

#     # 热力图（不叠底图，直接展示概率分布）
#     cellprob_vis = cv2.cvtColor(
#         cv2.applyColorMap((cellprob * 255).astype(np.uint8), cv2.COLORMAP_HOT),
#         cv2.COLOR_BGR2RGB)

#     # 实例掩膜（随机彩色填充）
#     masks_vis = draw_masks_on_gray(it_gray, pred_mask, alpha=0.75)

#     has_gt = gt_instance is not None
#     ncols  = 5 if has_gt else 4

#     fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

#     axes[0].imshow(ph_c);                    axes[0].set_title('Phase (RGB)');            axes[0].axis('off')
#     axes[1].imshow(it_gray, cmap='gray');    axes[1].set_title('Intensity (Gray)');       axes[1].axis('off')
#     axes[2].imshow(cellprob_vis);            axes[2].set_title('Cell Probability');        axes[2].axis('off')
#     axes[3].imshow(masks_vis);               axes[3].set_title(f'Predicted Masks ({pred_mask.max()} cells)'); axes[3].axis('off')

#     if has_gt:
#         axes[4].imshow(draw_masks_on_gray(it_gray, gt_instance, alpha=0.75))
#         axes[4].set_title(f'GT Masks ({gt_instance.max()} instances)')
#         axes[4].axis('off')

#     plt.suptitle(f'{stem}  |  {subdir}', fontsize=10)
#     plt.tight_layout()
#     return fig


# # ═══════════════════════════════════════════════════════════════
# # 单样本推理与可视化
# # ═══════════════════════════════════════════════════════════════

# DEMO_IDX = 5   # ← 修改为 0 ~ len(test_samples)-1

# s = test_samples[DEMO_IDX]
# print(f'Sample : {s["stem"]}')
# print(f'SubDir : {s["subdir"]}')

# ph_arr = read_rgb(s['phase'])
# it_arr = read_rgb(s['intensity'])
# mk_arr = read_mask(s['mask'])

# eval_model.eval()
# logit_full, ph_full, it_full = sliding_window_inference(eval_model, ph_arr, it_arr)

# # v8：pred_to_mask 直接返回 final_mask
# pred_mask = pred_to_mask(logit_full)
# cellprob  = sigmoid(logit_full[0])           # [H, W]

# gt_instance = None
# if mk_arr is not None:
#     gt_instance, _ = ndimage.label((mk_arr > 0).astype(np.uint8))

# fig = visualize_sample(ph_full, it_full, cellprob, pred_mask,
#                        gt_instance=gt_instance,
#                        stem=s['stem'], subdir=s['subdir'])
# plt.show()

# print(f'Predicted cells : {pred_mask.max()}')
# if gt_instance is not None:
#     print(f'GT instances    : {gt_instance.max()}')


## Step 9 — 批量推理并保存对比图

**v8 变更：**
- 推理结果保存到 `OUTPUT_DIR/<exp_id>/inference/`（与 eval/ 完全分离）
- 对比图 5 列（有 GT）/ 4 列（无 GT）
- `batch_summary.csv` 保存到 `inference/` 根目录

**inference/ 目录下文件：**
```
inference/
├── batch_summary.csv             所有样本细胞计数汇总
└── <subdir>/
    ├── <stem>_compare.png        对比图（5列有GT / 4列无GT）
    └── <stem>_pred_mask.tiff     最终实例 mask uint16
```


In [ ]:
# import pandas as pd
# import glob as _glob

# # ── ★ v7：指定推理哪个实验（仿照 Step 7 EVAL_EXP_ID 模式）────────
# # 留空 → 自动使用 Step 7/5 产生的 exp_id，或扫描最新 best.pth
# # 填写 → 强制推理该实验，适合跳过训练/评估直接批量推理
# if INFER_EXP_ID.strip():
#     # 模式 1：手动指定推理实验
#     exp_id       = INFER_EXP_ID.strip()
#     exp_ckpt_dir = os.path.join(CKPT_DIR,   exp_id)
#     exp_dir      = os.path.join(LOG_DIR,     exp_id)
#     exp_out_dir  = os.path.join(OUTPUT_DIR,  exp_id)
#     os.makedirs(exp_out_dir, exist_ok=True)
#     print(f'  [INFER] Using specified exp_id: {exp_id}')
# elif 'exp_ckpt_dir' not in dir():
#     # 模式 2：Step 5/7 未运行，自动扫描最新 best.pth
#     _cands = sorted(_glob.glob(os.path.join(CKPT_DIR, '*', 'best.pth')),
#                     key=os.path.getmtime, reverse=True)
#     if _cands:
#         _ck_probe    = torch.load(_cands[0], map_location='cpu')
#         exp_id       = _ck_probe.get('exp_id', os.path.basename(os.path.dirname(_cands[0])))
#         exp_ckpt_dir = os.path.dirname(_cands[0])
#     else:
#         exp_id       = 'unknown_exp'
#         exp_ckpt_dir = CKPT_DIR
#     exp_dir     = os.path.join(LOG_DIR,    exp_id)
#     exp_out_dir = os.path.join(OUTPUT_DIR, exp_id)
#     os.makedirs(exp_out_dir, exist_ok=True)
#     print(f'  [INFER] Auto-detected exp_id: {exp_id}')
# else:
#     # 模式 3：Step 5/7 已运行，直接使用其 exp_id
#     print(f'  [INFER] Using exp_id from Step 5/7: {exp_id}')

# # ── ★ v7：加载推理模型 ────────────────────────────────────────────
# # INFER_EXP_ID 指定时强制重新加载对应实验的 best.pth
# # 否则复用 Step 7 已加载的 eval_model（避免重复加载）
# _inf_ckpt = os.path.join(exp_ckpt_dir, 'best.pth')
# if INFER_EXP_ID.strip() and os.path.exists(_inf_ckpt):
#     eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
#     _ck = torch.load(_inf_ckpt, map_location=device)
#     eval_model.load_state_dict(_ck['model'])
#     print(f'  Loaded {_inf_ckpt}')
#     print(f'  epoch={_ck["epoch"]}  val_loss={_ck["best_loss"]:.6f}')
# elif 'eval_model' not in dir():
#     if os.path.exists(_inf_ckpt):
#         eval_model = DualBranchTransformer(pretrained_path=None, dtype=torch.float32).to(device)
#         _ck = torch.load(_inf_ckpt, map_location=device)
#         eval_model.load_state_dict(_ck['model'])
#         print(f'  Loaded {_inf_ckpt}')
#         print(f'  epoch={_ck["epoch"]}  val_loss={_ck["best_loss"]:.6f}')
#     else:
#         print(f'  [INFER WARNING] best.pth not found at {_inf_ckpt}, using current model.')
#         eval_model = model
# else:
#     print(f'  [INFER] Reusing eval_model from Step 7/5')

# print(f'Batch inference on {len(test_samples)} samples ...')
# eval_model.eval()
# summary = []

# # ★ v6：推理结果保存到 inference/ 子目录，与 eval/ 分离
# exp_inf_dir = os.path.join(exp_out_dir, 'inference')
# os.makedirs(exp_inf_dir, exist_ok=True)
# print(f'  Inference output dir: {exp_inf_dir}')

# for idx, s in enumerate(test_samples):
#     ph_arr = read_rgb(s['phase'])
#     it_arr = read_rgb(s['intensity'])
#     mk_arr = read_mask(s['mask'])

#     # 滑动窗口推理
#     logit_full, ph_full, it_full = sliding_window_inference(eval_model, ph_arr, it_arr)

#     # v8：pred_to_mask 直接返回 final_mask
#     pred_mask = pred_to_mask(logit_full)
#     cellprob  = sigmoid(logit_full[0])

#     gt_instance, n_gt = None, 0
#     if mk_arr is not None:
#         gt_instance, n_gt = ndimage.label((mk_arr > 0).astype(np.uint8))

#     fig = visualize_sample(ph_full, it_full, cellprob, pred_mask,
#                            gt_instance=gt_instance,
#                            stem=s['stem'], subdir=s['subdir'])

#     # 保存到 inference/<subdir>/
#     save_dir = os.path.join(exp_inf_dir, s['subdir'])
#     os.makedirs(save_dir, exist_ok=True)

#     fig.savefig(os.path.join(save_dir, s['stem'] + '_compare.png'),
#                 dpi=120, bbox_inches='tight')
#     plt.close(fig)

#     tifffile.imwrite(os.path.join(save_dir, s['stem'] + '_pred_mask.tiff'),
#                      pred_mask.astype(np.uint16))

#     summary.append({
#         'stem'       : s['stem'],
#         'subdir'     : s['subdir'],
#         'pred_cells' : int(pred_mask.max()),
#         'gt_cells'   : n_gt,
#     })

#     if (idx + 1) % 10 == 0:
#         print(f'  [{idx+1}/{len(test_samples)}] done ...')

# df_sum = pd.DataFrame(summary)
# # ★ v6：batch_summary.csv 保存到 inference/（不再是 exp_out_dir 根目录）
# _sum_path = os.path.join(exp_inf_dir, 'batch_summary.csv')
# df_sum.to_csv(_sum_path, index=False)

# print(f'\n✅ Done.  Processed: {len(summary)} images')
# print(f'   _compare.png + _pred_mask.tiff → {exp_inf_dir}')
# print(f'   batch_summary.csv              → {_sum_path}')
# print(f'\nCells per subdirectory:')
# print(df_sum.groupby('subdir')['pred_cells'].agg(['mean','min','max']).round(1))

